In [ ]:
import pandas as pd

# 1. Cargar el dataset de Airbnb
ruta_datos = "data/raw/airbnb/insideairbnb_barcelona_2026-06-24_listings.csv"
df = pd.read_csv(ruta_datos, low_memory=False)

# Total de registros
total_registros = len(df)
print(f"Total de registros cargados: {total_registros:,}")

In [ ]:
# 2. Clasificación de licencias según tipo de registro (HUTB/TU, NT, HB, AJ y nulas/sin número)

def clasificar_licencia(row):
    """
    Clasifica el registro según el tipo de licencia utilizando las reglas de negocio:
    - HUTB/HUT o código TU (ej. ESFCTU): Apartamento turístico (HUTB / TU).
    - Código NT (ej. ESFCNT): No turístico (Alquiler de temporada / NT).
    - Código HB: Licencia hotelera (HB).
    - Código AJ: Licencia albergue (AJ).
    - Texto de exención sin número de registro real o campo nulo: Sin licencia / Nula.
    - Otros códigos registrados.
    
    Args:
        row (pd.Series): Fila del DataFrame con el campo 'license'.
        
    Returns:
        str: Categoría asignada al tipo de licencia.
    """
    lic_val = row["license"]
    if pd.isna(lic_val) or str(lic_val).strip() in ["", "nan"]:
        return "Sin licencia / Nula"
    
    lic = str(lic_val)
    lic_upper = lic.upper()
    has_digits = any(c.isdigit() for c in lic)
    
    # 1. Códigos turísticos: HUTB/HUT o código nacional TU (ej. ESFCTU..., ESHFTU...)
    if "HUTB" in lic_upper or "HUT" in lic_upper or "ESFCTU" in lic_upper or "ESHFTU" in lic_upper:
        return "Apartamento turístico (HUTB / TU)"
        
    # 2. Códigos no turísticos: NT (ej. ESFCNT..., ESHFNT...)
    if "ESFCNT" in lic_upper or "ESHFNT" in lic_upper or ("NT" in lic_upper and has_digits):
        return "No turístico (Alquiler de temporada / NT)"
        
    # 3. Licencia hotelera (HB)
    if "HB-" in lic_upper or ("HB" in lic_upper and has_digits):
        return "Licencia hotelera (HB)"
        
    # 4. Licencia albergue (AJ)
    if "AJ" in lic_upper and has_digits:
        return "Licencia albergue (AJ)"
        
    # 5. Texto plano de exención sin número de registro real (ej. 'Exempt - seasonal rental')
    if not has_digits or "EXEMPT" in lic_upper:
        return "Sin licencia / Nula"
        
    return "Otras licencias registradas"

df["categoria_licencia"] = df.apply(clasificar_licencia, axis=1)

# Tabla de resumen general con Total y Porcentaje (%)
resumen = df["categoria_licencia"].value_counts().reset_index()
resumen.columns = ["Categoría de Licencia", "Total Registros"]
resumen["Porcentaje (%)"] = (resumen["Total Registros"] / total_registros * 100).round(2)

print("--- DESGLOSE TOTAL DE LICENCIAS Y PORCENTAJE (%) ---")
display(resumen)

In [ ]:
# 3. Análisis cruzado por categoría de licencia y tipo de habitación (room_type)

def analizar_licencias_por_tipo_habitacion(df_datos):
    """
    Analiza el desglose de categorías de licencia cruzado exclusivamente con el tipo
    de habitación ('room_type') y muestra los textos de licencia más habituales.
    
    Args:
        df_datos (pd.DataFrame): DataFrame con la columna 'categoria_licencia' y 'room_type'.
        
    Returns:
        pd.DataFrame: Tabla cruzada de categorías de licencia por room_type.
    """
    print("--- 1. Valores más frecuentes en la columna 'license' ---")
    display(df_datos["license"].value_counts(dropna=False).head(15).reset_index())
    
    print("\n--- 2. Desglose por categoría de licencia y tipo de habitación ('room_type') ---")
    desglose = pd.crosstab(
        df_datos["categoria_licencia"], 
        df_datos["room_type"], 
        margins=True, 
        margins_name="Total"
    ).sort_values(by="Total", ascending=False)
    
    display(desglose)
    
    return desglose

# Ejecutar análisis cruzado
tabla_desglose = analizar_licencias_por_tipo_habitacion(df)


In [ ]:
# 4. Filtrado de apartamentos turísticos y sin licencia sin comentarios en los últimos 12 meses

def analizar_inactivos_12m(df_datos):
    """
    Filtra las categorías 'Apartamento turístico (HUTB / TU)' y 'Sin licencia / Nula',
    excluyendo los registros que tengan comentarios en los últimos 12 meses
    (number_of_reviews_ltm > 0).
    
    Calcula los totales y porcentajes combinados ('de ambos') y por separado.
    
    Args:
        df_datos (pd.DataFrame): DataFrame original con 'categoria_licencia' y 'number_of_reviews_ltm'.
        
    Returns:
        pd.DataFrame: DataFrame filtrado con los registros sin comentarios en 12m.
    """
    # Categorías de interés
    categorias_interes = ["Apartamento turístico (HUTB / TU)", "Sin licencia / Nula"]
    
    # Filtrar solo esas 2 categorías
    df_interes = df_datos[df_datos["categoria_licencia"].isin(categorias_interes)].copy()
    
    # Quitar registros con comentarios en los últimos 12 meses (conservar number_of_reviews_ltm == 0)
    df_sin_resenas = df_interes[df_interes["number_of_reviews_ltm"] == 0].copy()
    
    total_general = len(df_datos)
    total_interes = len(df_interes)
    total_sin_resenas_ambos = len(df_sin_resenas)
    
    print("--- 1. RESUMEN COMBINADO (AMBOS GRUPOS SIN RESEÑAS EN 12 MESES) ---")
    print(f"Total combinado sin comentarios en 12m: {total_sin_resenas_ambos:,}")
    print(f"Porcentaje sobre las 2 categorías ({total_interes:,}): {(total_sin_resenas_ambos / total_interes * 100):.2f}%")
    print(f"Porcentaje sobre el total del dataset ({total_general:,}): {(total_sin_resenas_ambos / total_general * 100):.2f}%\n")
    
    # Desglose por separado
    registros_separado = []
    for cat in categorias_interes:
        tot_cat_original = len(df_datos[df_datos["categoria_licencia"] == cat])
        tot_cat_sin_resenas = len(df_sin_resenas[df_sin_resenas["categoria_licencia"] == cat])
        pct_sobre_categoria = (tot_cat_sin_resenas / tot_cat_original * 100) if tot_cat_original > 0 else 0
        pct_sobre_dataset = (tot_cat_sin_resenas / total_general * 100)
        
        registros_separado.append({
            "Categoría": cat,
            "Sin comentarios (12m)": tot_cat_sin_resenas,
            "Total categoría": tot_cat_original,
            "% sobre su categoría": round(pct_sobre_categoria, 2),
            "% sobre total dataset": round(pct_sobre_dataset, 2)
        })
        
    tabla_separado = pd.DataFrame(registros_separado)
    print("--- 2. DESGLOSE POR SEPARADO ---")
    display(tabla_separado)
    
    return df_sin_resenas

# Ejecutar análisis de inactivos en los últimos 12 meses
df_inactivos_12m = analizar_inactivos_12m(df)


In [ ]:
# 5. Detección de duplicados literales en alojamientos activos (con comentarios en los últimos 12 meses)

def analizar_duplicados_activos_12m(df_datos):
    """
    Filtra los alojamientos de las categorías 'Apartamento turístico (HUTB / TU)' y
    'Sin licencia / Nula' que hayan estado activos en los últimos 12 meses (number_of_reviews_ltm > 0).
    
    Identifica duplicados literales bajo diferentes criterios (identidad exacta, mismo nombre/anfitrión,
    y misma licencia), mostrando los totales y porcentajes agrupados (juntos) y por separado.
    
    Args:
        df_datos (pd.DataFrame): DataFrame original con las categorías de licencia y número de reseñas.
        
    Returns:
        pd.DataFrame: DataFrame filtrado con el universo activo analizado.
    """
    categorias_interes = ["Apartamento turístico (HUTB / TU)", "Sin licencia / Nula"]
    
    # 1. Filtrar universo activo (con reseñas en los últimos 12 meses)
    mask_activos = df_datos["categoria_licencia"].isin(categorias_interes) & (df_datos["number_of_reviews_ltm"] > 0)
    df_activos = df_datos[mask_activos].copy()
    
    tot_general_activos = len(df_activos)
    tot_apt_activos = len(df_activos[df_activos["categoria_licencia"] == "Apartamento turístico (HUTB / TU)"])
    tot_sin_activos = len(df_activos[df_activos["categoria_licencia"] == "Sin licencia / Nula"])
    
    print("--- 1. UNIVERSO ACTIVO (CON COMENTARIOS EN ÚLTIMOS 12 MESES) ---")
    print(f"Total activo combinado (ambas categorías): {tot_general_activos:,}")
    print(f"  - Apartamentos turísticos (HUTB / TU): {tot_apt_activos:,}")
    print(f"  - Sin licencia / Nula: {tot_sin_activos:,}\n")
    
    # 2. Criterios de duplicación
    criterios = {
        "Duplicados exactos por identidad (nombre, anfitrión, coords, tipo, precio)": ["name", "host_id", "latitude", "longitude", "room_type", "price"],
        "Duplicados por mismo nombre y anfitrión (name, host_id)": ["name", "host_id"],
        "Duplicados por misma licencia (código repetido)": ["license"]
    }
    
    resultados_duplicados = []
    
    for nombre_criterio, columnas_criterio in criterios.items():
        if columnas_criterio == ["license"]:
            mask_dup = df_activos["license"].notna() & df_activos.duplicated(subset=columnas_criterio, keep=False)
        else:
            mask_dup = df_activos.duplicated(subset=columnas_criterio, keep=False)
            
        df_dup = df_activos[mask_dup]
        
        n_juntos = len(df_dup)
        pct_juntos = (n_juntos / tot_general_activos * 100) if tot_general_activos > 0 else 0
        
        n_apt = len(df_dup[df_dup["categoria_licencia"] == "Apartamento turístico (HUTB / TU)"])
        pct_apt = (n_apt / tot_apt_activos * 100) if tot_apt_activos > 0 else 0
        
        n_sin = len(df_dup[df_dup["categoria_licencia"] == "Sin licencia / Nula"])
        pct_sin = (n_sin / tot_sin_activos * 100) if tot_sin_activos > 0 else 0
        
        resultados_duplicados.append({
            "Criterio de Duplicación": nombre_criterio,
            "Total Juntos": n_juntos,
            "% Juntos": round(pct_juntos, 2),
            "Apt. Turístico": n_apt,
            "% Apt. Turístico": round(pct_apt, 2),
            "Sin Licencia": n_sin,
            "% Sin Licencia": round(pct_sin, 2)
        })
        
    tabla_duplicados = pd.DataFrame(resultados_duplicados)
    
    print("--- 2. ANÁLISIS DE DUPLICADOS LITERALES (TOTALES Y % JUNTOS Y POR SEPARADO) ---")
    display(tabla_duplicados)
    
    return df_activos

# Ejecutar análisis de duplicados en el universo activo
df_universo_activo = analizar_duplicados_activos_12m(df)


In [ ]:
# 6. Desglose de duplicados por combinaciones clave (Nombre+Anfitrión, Licencia, Anfitrión+Licencia, y Anfitrión+Licencia+Precio)

def desglosar_duplicados_por_tipo(df_datos):
    """
    Analiza los alojamientos duplicados según diversas combinaciones de campos:
    - Nombre + Anfitrión (name, host_id).
    - Misma Licencia (license).
    - Nombre + Anfitrión + Licencia (coincidencia triple).
    - Anfitrión + Licencia (host_id, license).
    - Anfitrión + Licencia + Precio (host_id, license, price).
    
    Muestra dos tablas separadas desglosadas por 'room_type':
    - Tabla 1: Apartamentos turísticos (HUTB / TU).
    - Tabla 2: Sin licencia / Nula.
    
    Args:
        df_datos (pd.DataFrame): DataFrame original de Airbnb.
        
    Returns:
        tuple: (pd.DataFrame tabla_apt_turisticos, pd.DataFrame tabla_sin_licencia)
    """
    categorias_interes = ["Apartamento turístico (HUTB / TU)", "Sin licencia / Nula"]
    
    # Universo activo (con reseñas en los últimos 12 meses)
    mask_activos = df_datos["categoria_licencia"].isin(categorias_interes) & (df_datos["number_of_reviews_ltm"] > 0)
    df_activos = df_datos[mask_activos].copy()
    
    # Criterio 1: Mismo nombre y anfitrión (name, host_id)
    df_activos["dup_nombre_anfitrion"] = df_activos.duplicated(subset=["name", "host_id"], keep=False)
    
    # Criterio 2: Misma licencia (license, excluyendo nulos)
    df_activos["dup_misma_licencia"] = df_activos["license"].notna() & df_activos.duplicated(subset=["license"], keep=False)
    
    # Criterio 3: Coincidencia Triple (name, host_id, license)
    df_activos["dup_triple"] = df_activos["license"].notna() & df_activos.duplicated(subset=["name", "host_id", "license"], keep=False)
    
    # Criterio 4: Anfitrión + Licencia (host_id, license)
    df_activos["dup_host_lic"] = df_activos["license"].notna() & df_activos.duplicated(subset=["host_id", "license"], keep=False)
    
    # Criterio 5: Anfitrión + Licencia + Precio (host_id, license, price)
    df_activos["dup_host_lic_price"] = df_activos["license"].notna() & df_activos.duplicated(subset=["host_id", "license", "price"], keep=False)
    
    col_agrupacion = ["property_type", "room_type"] if "property_type" in df_datos.columns else ["room_type"]
    
    tablas_resultado = {}
    
    for cat in categorias_interes:
        df_cat = df_activos[df_activos["categoria_licencia"] == cat]
        
        # Agrupar recuentos por criterio
        counts_nh = df_cat[df_cat["dup_nombre_anfitrion"]].groupby(col_agrupacion).size()
        counts_lic = df_cat[df_cat["dup_misma_licencia"]].groupby(col_agrupacion).size()
        counts_triple = df_cat[df_cat["dup_triple"]].groupby(col_agrupacion).size()
        counts_hl = df_cat[df_cat["dup_host_lic"]].groupby(col_agrupacion).size()
        counts_hlp = df_cat[df_cat["dup_host_lic_price"]].groupby(col_agrupacion).size()
        
        tipos_unicos = df_cat[col_agrupacion].drop_duplicates().sort_values(by=col_agrupacion)
        tabla = tipos_unicos.copy()
        
        if len(col_agrupacion) == 1:
            col_key = col_agrupacion[0]
            tabla["Duplicados Mismo Nombre + Anfitrión"] = tabla[col_key].map(counts_nh).fillna(0).astype(int)
            tabla["Duplicados Misma Licencia"] = tabla[col_key].map(counts_lic).fillna(0).astype(int)
            tabla["Duplicados Mismo Nombre + Anfitrión + Licencia"] = tabla[col_key].map(counts_triple).fillna(0).astype(int)
            tabla["Duplicados Anfitrión + Licencia"] = tabla[col_key].map(counts_hl).fillna(0).astype(int)
            tabla["Duplicados Anfitrión + Licencia + Precio"] = tabla[col_key].map(counts_hlp).fillna(0).astype(int)
        else:
            tabla["Duplicados Mismo Nombre + Anfitrión"] = tabla.set_index(col_agrupacion).index.map(counts_nh).fillna(0).astype(int)
            tabla["Duplicados Misma Licencia"] = tabla.set_index(col_agrupacion).index.map(counts_lic).fillna(0).astype(int)
            tabla["Duplicados Mismo Nombre + Anfitrión + Licencia"] = tabla.set_index(col_agrupacion).index.map(counts_triple).fillna(0).astype(int)
            tabla["Duplicados Anfitrión + Licencia"] = tabla.set_index(col_agrupacion).index.map(counts_hl).fillna(0).astype(int)
            tabla["Duplicados Anfitrión + Licencia + Precio"] = tabla.set_index(col_agrupacion).index.map(counts_hlp).fillna(0).astype(int)
            
        tablas_resultado[cat] = tabla
        
    print("================ TABLA 1: APARTAMENTOS TURÍSTICOS (HUTB / TU) ================")
    display(tablas_resultado["Apartamento turístico (HUTB / TU)"])
    
    print("\n================ TABLA 2: SIN LICENCIA / NULA ================")
    display(tablas_resultado["Sin licencia / Nula"])
    
    return tablas_resultado["Apartamento turístico (HUTB / TU)"], tablas_resultado["Sin licencia / Nula"]

# Ejecutar desglose por tipo de habitación y propiedad en tablas separadas
tabla_apt_dup, tabla_sin_dup = desglosar_duplicados_por_tipo(df)


In [ ]:
# 7. Tabla 1: Duplicados por Mismo Nombre, Mismo Anfitrión, Misma Licencia y coincidencia de los tres (Apartamentos turísticos)

def analizar_duplicados_exactos_turistico(df_datos):
    """
    Muestra para 'Entire home/apt' y 'Private room' los anuncios duplicados por:
    - Mismo Nombre (títulos iguales entre cualquier anuncio).
    - Mismo Anfitrión (anuncios duplicados publicados por el mismo host: name + host_id).
    - Misma Licencia (código de licencia repetido).
    - Coincidencia Triple (mismo nombre, anfitrión y licencia a la vez).
    
    Args:
        df_datos (pd.DataFrame): DataFrame original de Airbnb.
        
    Returns:
        pd.DataFrame: Tabla 1 corregida centrada en anuncios duplicados.
    """
    mask_activos = (df_datos["categoria_licencia"] == "Apartamento turístico (HUTB / TU)") & (df_datos["number_of_reviews_ltm"] > 0)
    df_apt = df_datos[mask_activos].copy()
    
    filas_t1 = []
    for rtype in ["Entire home/apt", "Private room"]:
        sub = df_apt[df_apt["room_type"] == rtype].copy()
        filas_t1.append({
            "Tipo de Habitación": rtype,
            "Duplicados Mismo Nombre": sub.duplicated(subset=["name"], keep=False).sum(),
            "Duplicados Mismo Anfitrión (Nombre + Host)": sub.duplicated(subset=["name", "host_id"], keep=False).sum(),
            "Duplicados Misma Licencia": (sub["license"].notna() & sub.duplicated(subset=["license"], keep=False)).sum(),
            "Coincidencia Triple (Nombre + Host + Licencia)": (sub["license"].notna() & sub.duplicated(subset=["name", "host_id", "license"], keep=False)).sum()
        })
        
    tabla_1 = pd.DataFrame(filas_t1)
    print("================ TABLA 1: APARTAMENTOS TURÍSTICOS (HUTB / TU) ================")
    display(tabla_1)
    return tabla_1

# Ejecutar Tabla 1 corregida
tabla_1_resultado = analizar_duplicados_exactos_turistico(df)


In [ ]:
# 8. Tabla 2: Hosts con un mismo anuncio repetido (Alojamientos sin licencia / nula)

def analizar_hosts_mismo_anuncio(df_datos):
    """
    Muestra para 'Private room' y 'Entire home/apt' en alojamientos sin licencia cuántos hosts
    tienen un mismo anuncio exactamente repetido (mismo nombre y anfitrión) y cuántos anuncios suman.
    
    Args:
        df_datos (pd.DataFrame): DataFrame original de Airbnb.
        
    Returns:
        pd.DataFrame: Tabla 2 con la relación de hosts y anuncios repetidos.
    """
    mask_activos = (df_datos["categoria_licencia"] == "Sin licencia / Nula") & (df_datos["number_of_reviews_ltm"] > 0)
    df_sin = df_datos[mask_activos].copy()
    
    filas_t2 = []
    for rtype in ["Private room", "Entire home/apt"]:
        sub = df_sin[df_sin["room_type"] == rtype].copy()
        dup_name_host = sub.duplicated(subset=["name", "host_id"], keep=False)
        hosts_count = sub[dup_name_host]["host_id"].nunique()
        anuncios_count = dup_name_host.sum()
        
        filas_t2.append({
            "Tipo de Habitación": rtype,
            "Hosts con Mismo Anuncio": hosts_count,
            "Anuncios Repetidos (Mismo Nombre + Host)": anuncios_count
        })
        
    tabla_2 = pd.DataFrame(filas_t2)
    print("================ TABLA 2: SIN LICENCIA / NULA ================")
    display(tabla_2)
    return tabla_2

# Ejecutar Tabla 2
tabla_2_resultado = analizar_hosts_mismo_anuncio(df)


In [ ]:
# 9. Desduplicación progresiva acumulada (incluyendo eliminación en Sin Licencia por Mismo Precio+Nombre+Anfitrión)

def aplicar_desduplicacion_progresiva(df_datos):
    """
    Toma los alojs. activos de 'Apartamento turístico (HUTB / TU)' y 'Sin licencia / Nula'.
    1. Conserva solo 1 registro para duplicados en 'Anfitrión + Licencia + Precio'.
    2. Conserva solo 1 registro para duplicados en 'Nombre + Anfitrión + Licencia'.
    3. Conserva solo 1 registro para duplicados en 'Anfitrión + Licencia'.
    4. Conserva solo 1 registro para duplicados en Sin Licencia con 'Precio + Nombre + Anfitrión'.
    5. Recalcula las tablas para observar el impacto en los conteos de duplicados.
    
    Args:
        df_datos (pd.DataFrame): DataFrame original de Airbnb.
        
    Returns:
        tuple: (pd.DataFrame df_desduplicado, dict tablas_resultantes)
    """
    categorias_interes = ["Apartamento turístico (HUTB / TU)", "Sin licencia / Nula"]
    mask_activos = df_datos["categoria_licencia"].isin(categorias_interes) & (df_datos["number_of_reviews_ltm"] > 0)
    df_activos = df_datos[mask_activos].copy()
    
    tot_inicial = len(df_activos)
    
    # 1. Desduplicar 'Anfitrión + Licencia + Precio'
    mask_lic_1 = df_activos["license"].notna()
    df_lic_1 = df_activos[mask_lic_1].drop_duplicates(subset=["host_id", "license", "price"], keep="first")
    df_paso_1 = pd.concat([df_lic_1, df_activos[~mask_lic_1]], ignore_index=True)
    
    # 2. Desduplicar 'Nombre + Anfitrión + Licencia'
    mask_lic_2 = df_paso_1["license"].notna()
    df_lic_2 = df_paso_1[mask_lic_2].drop_duplicates(subset=["name", "host_id", "license"], keep="first")
    df_paso_2 = pd.concat([df_lic_2, df_paso_1[~mask_lic_2]], ignore_index=True)
    
    # 3. Desduplicar 'Anfitrión + Licencia'
    mask_lic_3 = df_paso_2["license"].notna()
    df_lic_3 = df_paso_2[mask_lic_3].drop_duplicates(subset=["host_id", "license"], keep="first")
    df_paso_3 = pd.concat([df_lic_3, df_paso_2[~mask_lic_3]], ignore_index=True)
    
    # 4. Desduplicar Sin Licencia por 'Precio + Nombre + Anfitrión'
    mask_sin = df_paso_3["categoria_licencia"] == "Sin licencia / Nula"
    df_sin_dedup = df_paso_3[mask_sin].drop_duplicates(subset=["price", "name", "host_id"], keep="first")
    df_dedup = pd.concat([df_paso_3[~mask_sin], df_sin_dedup], ignore_index=True)
    
    tot_final = len(df_dedup)
    eliminados = tot_inicial - tot_final
    
    print(f"--- PROCESO DE DESDUPLICACIÓN ACUMULADO --- ")
    print(f"Registros activos iniciales: {tot_inicial:,}")
    print(f"Registros activos tras desduplicar Sin Licencia por Precio+Nombre+Host: {tot_final:,} (se eliminaron {eliminados:,} duplicados sobrantes en total)\n")
    
    # Recalcular banderas de duplicidad
    df_dedup["dup_nombre_host"] = df_dedup.duplicated(subset=["name", "host_id"], keep=False)
    df_dedup["dup_licencia"] = df_dedup["license"].notna() & df_dedup.duplicated(subset=["license"], keep=False)
    df_dedup["dup_triple"] = df_dedup["license"].notna() & df_dedup.duplicated(subset=["name", "host_id", "license"], keep=False)
    df_dedup["dup_host_lic"] = df_dedup["license"].notna() & df_dedup.duplicated(subset=["host_id", "license"], keep=False)
    df_dedup["dup_host_lic_price"] = df_dedup["license"].notna() & df_dedup.duplicated(subset=["host_id", "license", "price"], keep=False)
    df_dedup["dup_precio_nombre_host"] = df_dedup.duplicated(subset=["price", "name", "host_id"], keep=False)
    
    col_agrupacion = ["property_type", "room_type"] if "property_type" in df_datos.columns else ["room_type"]
    tablas_resultado = {}
    
    for cat in categorias_interes:
        df_cat = df_dedup[df_dedup["categoria_licencia"] == cat]
        
        counts_nh = df_cat[df_cat["dup_nombre_host"]].groupby(col_agrupacion).size()
        counts_lic = df_cat[df_cat["dup_licencia"]].groupby(col_agrupacion).size()
        counts_triple = df_cat[df_cat["dup_triple"]].groupby(col_agrupacion).size()
        counts_hl = df_cat[df_cat["dup_host_lic"]].groupby(col_agrupacion).size()
        counts_hlp = df_cat[df_cat["dup_host_lic_price"]].groupby(col_agrupacion).size()
        
        tipos_unicos = df_cat[col_agrupacion].drop_duplicates().sort_values(by=col_agrupacion)
        tabla = tipos_unicos.copy()
        
        if len(col_agrupacion) == 1:
            col_key = col_agrupacion[0]
            tabla["Duplicados Mismo Nombre + Anfitrión"] = tabla[col_key].map(counts_nh).fillna(0).astype(int)
            tabla["Duplicados Misma Licencia"] = tabla[col_key].map(counts_lic).fillna(0).astype(int)
            tabla["Duplicados Mismo Nombre + Anfitrión + Licencia"] = tabla[col_key].map(counts_triple).fillna(0).astype(int)
            tabla["Duplicados Anfitrión + Licencia"] = tabla[col_key].map(counts_hl).fillna(0).astype(int)
            tabla["Duplicados Anfitrión + Licencia + Precio"] = tabla[col_key].map(counts_hlp).fillna(0).astype(int)
        else:
            tabla["Duplicados Mismo Nombre + Anfitrión"] = tabla.set_index(col_agrupacion).index.map(counts_nh).fillna(0).astype(int)
            tabla["Duplicados Misma Licencia"] = tabla.set_index(col_agrupacion).index.map(counts_lic).fillna(0).astype(int)
            tabla["Duplicados Mismo Nombre + Anfitrión + Licencia"] = tabla.set_index(col_agrupacion).index.map(counts_triple).fillna(0).astype(int)
            tabla["Duplicados Anfitrión + Licencia"] = tabla.set_index(col_agrupacion).index.map(counts_hl).fillna(0).astype(int)
            tabla["Duplicados Anfitrión + Licencia + Precio"] = tabla.set_index(col_agrupacion).index.map(counts_hlp).fillna(0).astype(int)
            
        tablas_resultado[cat] = tabla
        
    print("================ TABLA 1 (RECALCULADA): APARTAMENTOS TURÍSTICOS (HUTB / TU) ================")
    display(tablas_resultado["Apartamento turístico (HUTB / TU)"])
    
    print("\n================ TABLA 2 (RECALCULADA): SIN LICENCIA / NULA ================")
    display(tablas_resultado["Sin licencia / Nula"])
    
    return df_dedup, tablas_resultado

# Ejecutar desduplicación acumulada y recalcular tablas
df_activos_dedup, tablas_dedup = aplicar_desduplicacion_progresiva(df)


In [ ]:
# 10. Gráfico lineal comparativo de duplicados (incluyendo línea morada para Mismo Precio + Nombre + Anfitrión en Sin Licencia)

import matplotlib.pyplot as plt

def graficar_duplicados_lineal(df_desduplicado):
    """
    Genera los gráficos lineales comparativos para 'Entire home/apt' en Tabla 1 y Tabla 2:
    - Línea Roja: Duplicados por Mismo Nombre + Anfitrión.
    - Línea Azul: Duplicados por Anfitrión + Licencia.
    - Línea Verde: Duplicados por Mismo Nombre, Anfitrión y Licencia.
    - Línea Morada (solo en Tabla 2 Sin Licencia): Duplicados por Mismo Precio, Nombre y Anfitrión.
    
    Args:
        df_desduplicado (pd.DataFrame): DataFrame desduplicado obtenido en la celda 9.
    """
    categorias = ["Apartamento turístico (HUTB / TU)", "Sin licencia / Nula"]
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    for i, cat in enumerate(categorias):
        sub = df_desduplicado[(df_desduplicado["categoria_licencia"] == cat) & (df_desduplicado["room_type"] == "Entire home/apt")].copy()
        
        # 1. Grupos duplicados por Mismo Nombre + Anfitrión (Rojo)
        nh_sizes = sub[sub["dup_nombre_host"]].groupby(["name", "host_id"]).size()
        nh_dist = nh_sizes.value_counts().sort_index()
        
        # 2. Grupos duplicados por Anfitrión + Licencia (Azul)
        hl_sizes = sub[sub["dup_host_lic"]].groupby(["host_id", "license"]).size()
        hl_dist = hl_sizes.value_counts().sort_index()
        
        # 3. Grupos duplicados por Mismo Nombre + Anfitrión + Licencia (Verde)
        triple_sizes = sub[sub["dup_triple"]].groupby(["name", "host_id", "license"]).size()
        triple_dist = triple_sizes.value_counts().sort_index()
        
        # 4. Grupos duplicados por Mismo Precio + Nombre + Anfitrión (Morado)
        pnh_sizes = sub[sub["dup_precio_nombre_host"]].groupby(["price", "name", "host_id"]).size()
        pnh_dist = pnh_sizes.value_counts().sort_index()
        
        max_x = max(
            nh_dist.index.max() if len(nh_dist) > 0 else 2,
            hl_dist.index.max() if len(hl_dist) > 0 else 2,
            triple_dist.index.max() if len(triple_dist) > 0 else 2,
            pnh_dist.index.max() if len(pnh_dist) > 0 else 2
        )
        x_range = list(range(2, max_x + 1))
        
        y_nh = [nh_dist.get(x, 0) for x in x_range]
        y_hl = [hl_dist.get(x, 0) for x in x_range]
        y_triple = [triple_dist.get(x, 0) for x in x_range]
        y_pnh = [pnh_dist.get(x, 0) for x in x_range]
        
        axes[i].plot(x_range, y_nh, color="red", marker="o", linewidth=2, label="Mismo Nombre + Anfitrión")
        axes[i].plot(x_range, y_hl, color="blue", marker="s", linewidth=2, label="Anfitrión + Licencia")
        axes[i].plot(x_range, y_triple, color="green", marker="^", linewidth=2, label="Nombre + Anfitrión + Licencia")
        
        if cat == "Sin licencia / Nula":
            axes[i].plot(x_range, y_pnh, color="purple", marker="d", linewidth=2, label="Precio + Nombre + Anfitrión")
            
        axes[i].set_title(f"Grupos duplicados Entire home/apt - {cat}", fontsize=11)
        axes[i].set_xlabel("Número de anuncios duplicados por grupo")
        axes[i].set_ylabel("Cantidad de grupos duplicados")
        axes[i].legend()
        axes[i].grid(True, linestyle="--", alpha=0.6)
        
    plt.tight_layout()
    plt.show()

# Generar gráfico lineal comparativo actualizado
graficar_duplicados_lineal(df_activos_dedup)


In [ ]:
# 11. Tabla resumen final de duplicados tras todo el proceso de desduplicación (VUTs/TU y Sin Licencia)

def mostrar_tablas_finales_duplicados(df_desduplicado):
    """
    Genera y muestra en una celda dedicada las dos tablas resumen de duplicados finales 
    para 'Apartamento turístico (HUTB / TU)' y 'Sin licencia / Nula' desglosadas por 'room_type'.
    
    Args:
        df_desduplicado (pd.DataFrame): DataFrame limpio tras todo el proceso de desduplicación acumulado.
        
    Returns:
        tuple: (pd.DataFrame tabla_vut, pd.DataFrame tabla_sin_licencia)
    """
    categorias = ["Apartamento turístico (HUTB / TU)", "Sin licencia / Nula"]
    tablas = {}
    
    for cat in categorias:
        df_cat = df_desduplicado[df_desduplicado["categoria_licencia"] == cat]
        
        counts_nh = df_cat[df_cat["dup_nombre_host"]].groupby("room_type").size()
        counts_lic = df_cat[df_cat["dup_licencia"]].groupby("room_type").size()
        counts_triple = df_cat[df_cat["dup_triple"]].groupby("room_type").size()
        counts_hl = df_cat[df_cat["dup_host_lic"]].groupby("room_type").size()
        counts_hlp = df_cat[df_cat["dup_host_lic_price"]].groupby("room_type").size()
        
        tabla = pd.DataFrame({"Tipo de Habitación (room_type)": df_cat["room_type"].unique()}).sort_values(by="Tipo de Habitación (room_type)")
        tabla["Duplicados Mismo Nombre + Anfitrión"] = tabla["Tipo de Habitación (room_type)"].map(counts_nh).fillna(0).astype(int)
        tabla["Duplicados Misma Licencia"] = tabla["Tipo de Habitación (room_type)"].map(counts_lic).fillna(0).astype(int)
        tabla["Duplicados Nombre + Anfitrión + Licencia"] = tabla["Tipo de Habitación (room_type)"].map(counts_triple).fillna(0).astype(int)
        tabla["Duplicados Anfitrión + Licencia"] = tabla["Tipo de Habitación (room_type)"].map(counts_hl).fillna(0).astype(int)
        tabla["Duplicados Anfitrión + Licencia + Precio"] = tabla["Tipo de Habitación (room_type)"].map(counts_hlp).fillna(0).astype(int)
        
        tablas[cat] = tabla
        
    print("================ TABLA FINAL 1: APARTAMENTOS TURÍSTICOS (HUTB / TU - VUTs) ================")
    display(tablas["Apartamento turístico (HUTB / TU)"])
    
    print("\n================ TABLA FINAL 2: SIN LICENCIA / NULA ================")
    display(tablas["Sin licencia / Nula"])
    
    return tablas["Apartamento turístico (HUTB / TU)"], tablas["Sin licencia / Nula"]

# Ejecutar y visualizar las dos tablas resumen finales en esta nueva celda
tabla_vut_final, tabla_sin_final = mostrar_tablas_finales_duplicados(df_activos_dedup)


In [ ]:
# 12. Desduplicación secuencial en Tabla 1 (Roja por GPS y Precio, Azul por Licencia+GPS) y gráficos finales

import matplotlib.pyplot as plt
import pandas as pd

def aplicar_desduplicacion_tabla1(df_desduplicado):
    """
    1. Desduplica Tabla 2 (Sin Licencia) conservando 1 solo registro por anuncio real (Mismo Nombre + Anfitrión).
    2. Desduplica Tabla 1 (Apartamentos Turísticos VUTs) secuencialmente:
       - Línea ROJA por Coordenadas GPS (name, host_id, lat, lon).
       - Línea ROJA por Precio (name, host_id, price).
       - Línea AZUL por Licencia y GPS (license, lat, lon).
    3. Muestra las tablas y el gráfico comparativo de duplicados restantes.
    
    Args:
        df_desduplicado (pd.DataFrame): Dataset desduplicado acumulado de la celda 9.
        
    Returns:
        pd.DataFrame: Dataset final completamente limpio.
    """
    # 1. Limpieza total de Tabla 2 (Sin Licencia / Nula)
    mask_sin = df_desduplicado["categoria_licencia"] == "Sin licencia / Nula"
    df_sin_limpio = df_desduplicado[mask_sin].drop_duplicates(subset=["name", "host_id"], keep="first")
    df_paso1 = pd.concat([df_desduplicado[~mask_sin], df_sin_limpio], ignore_index=True)
    
    # 2. Desduplicación secuencial de Tabla 1 (Apartamentos Turísticos VUTs)
    mask_t1 = df_paso1["categoria_licencia"] == "Apartamento turístico (HUTB / TU)"
    df_t1 = df_paso1[mask_t1].copy()
    
    tot_t1_inicial = len(df_t1)
    
    # Paso A: Desduplicar ROJA por Nombre + Anfitrión + Coordenadas GPS
    df_t1_gps = df_t1.drop_duplicates(subset=["name", "host_id", "latitude", "longitude"], keep="first")
    
    # Paso B: Desduplicar ROJA por Nombre + Anfitrión + Precio
    df_t1_precio = df_t1_gps.drop_duplicates(subset=["name", "host_id", "price"], keep="first")
    
    # Paso C: Desduplicar AZUL por Licencia + Coordenadas GPS
    mask_lic_t1 = df_t1_precio["license"].notna()
    df_t1_lic_gps = df_t1_precio[mask_lic_t1].drop_duplicates(subset=["license", "latitude", "longitude"], keep="first")
    df_t1_dedup = pd.concat([df_t1_lic_gps, df_t1_precio[~mask_lic_t1]], ignore_index=True)
    
    tot_t1_final = len(df_t1_dedup)
    eliminados_t1 = tot_t1_inicial - tot_t1_final
    
    print(f"--- DESDUPLICACIÓN EN TABLA 1 (APARTAMENTOS TURÍSTICOS VUTs) ---")
    print(f"Registros en Tabla 1 iniciales: {tot_t1_inicial:,}")
    print(f"Registros en Tabla 1 tras desduplicar Roja por GPS y Precio, y Azul por Licencia+GPS: {tot_t1_final:,} (se eliminaron {eliminados_t1:,} duplicados sobrantes)\n")
    
    df_limpio_final = pd.concat([df_paso1[~mask_t1], df_t1_dedup], ignore_index=True)
    
    # Recalcular banderas de duplicidad en Tabla 1
    df_t1_dedup["dup_nombre_host"] = df_t1_dedup.duplicated(subset=["name", "host_id"], keep=False)
    df_t1_dedup["dup_licencia"] = df_t1_dedup["license"].notna() & df_t1_dedup.duplicated(subset=["license"], keep=False)
    
    # Subconjuntos restantes
    df_nh = df_t1_dedup[df_t1_dedup["dup_nombre_host"]]
    df_lic = df_t1_dedup[df_t1_dedup["dup_licencia"]]
    
    var_nh = {
        "Mismo Barrio": df_nh.duplicated(subset=["name", "host_id", "neighbourhood"], keep=False).sum(),
        "Mismo Mínimo de Noches": df_nh.duplicated(subset=["name", "host_id", "minimum_nights"], keep=False).sum(),
        "Mismas Coordenadas GPS": df_nh.duplicated(subset=["name", "host_id", "latitude", "longitude"], keep=False).sum(),
        "Mismo Precio": df_nh.duplicated(subset=["name", "host_id", "price"], keep=False).sum()
    }
    
    var_lic = {
        "Mismo Barrio": df_lic.duplicated(subset=["license", "neighbourhood"], keep=False).sum(),
        "Mismo Mínimo de Noches": df_lic.duplicated(subset=["license", "minimum_nights"], keep=False).sum(),
        "Mismas Coordenadas GPS": df_lic.duplicated(subset=["license", "latitude", "longitude"], keep=False).sum(),
        "Mismo Precio": df_lic.duplicated(subset=["license", "price"], keep=False).sum()
    }
    
    # Gráficos
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    s_nh = pd.Series(var_nh).sort_values(ascending=True)
    s_lic = pd.Series(var_lic).sort_values(ascending=True)
    
    tot_nh = len(df_nh)
    tot_lic = len(df_lic)
    
    b1 = axes[0].barh(s_nh.index, s_nh.values, color="crimson", edgecolor="darkred")
    axes[0].set_title(f"Tabla 1: Duplicados Mismo Nombre + Anfitrión ({tot_nh} restantes)\nCoincidencias en otras variables", fontsize=11)
    axes[0].set_xlabel("Número de anuncios duplicados que coinciden")
    axes[0].grid(axis="x", linestyle="--", alpha=0.6)
    for b in b1:
        w = b.get_width()
        axes[0].text(w + 1, b.get_y() + b.get_height()/2, f"{int(w)}", va="center", fontweight="bold")
        
    b2 = axes[1].barh(s_lic.index, s_lic.values, color="dodgerblue", edgecolor="navy")
    axes[1].set_title(f"Tabla 1: Duplicados Misma Licencia ({tot_lic} restantes)\nCoincidencias en otras variables", fontsize=11)
    axes[1].set_xlabel("Número de anuncios duplicados que coinciden")
    axes[1].grid(axis="x", linestyle="--", alpha=0.6)
    for b in b2:
        w = b.get_width()
        axes[1].text(w + 1, b.get_y() + b.get_height()/2, f"{int(w)}", va="center", fontweight="bold")
        
    plt.tight_layout()
    plt.show()
    
    return df_limpio_final

# Ejecutar desduplicaciones secuenciales en Tabla 1 y generar gráficos
df_limpio_final = aplicar_desduplicacion_tabla1(df_activos_dedup)


In [ ]:
# 13. Mapa de Barcelona con límites de barrios (sin nombres), geolocalización (Cuadrados=Pisos, Círculos=Habitaciones) y < 200m

import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def haversine_m(lat1, lon1, lat2, lon2):
    """Calcula la distancia geodésica en metros entre dos coordenadas (Haversine)."""
    R = 6371000  # Radio terrestre en metros
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2.0) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

def geolocalizar_duplicados_200m(df_limpio):
    """
    1. Carga las geometrías de los barrios de Barcelona (GeoJSON).
    2. Identifica los alojs. duplicados de la Tabla 1 (VUTs) y calcula cuáles se ubican a < 200 metros.
    3. Dibuja el mapa completo sin etiquetas de texto de nombres de barrios:
       - Límites perimetrales de los barrios.
       - Cuadrados ('s'): Apartamentos enteros ('Entire home/apt').
       - Círculos ('o'): Habitaciones privadas ('Private room').
       - Color Rojo/Naranja: Distancia < 200 metros.
       - Color Azul/Cian: Distancia >= 200 metros.
    
    Args:
        df_limpio (pd.DataFrame): Dataset limpio tras las desduplicaciones de la celda 12.
    """
    # Cargar GeoJSON de barrios de Barcelona
    ruta_geojson = "data/raw/geometria/insideairbnb_barrios_barcelona.geojson"
    with open(ruta_geojson, "r", encoding="utf-8") as f:
        geo_barrios = json.load(f)
        
    mask_t1 = df_limpio["categoria_licencia"] == "Apartamento turístico (HUTB / TU)"
    df_t1 = df_limpio[mask_t1].copy()
    
    df_t1["dup_nombre_host"] = df_t1.duplicated(subset=["name", "host_id"], keep=False)
    df_t1["dup_licencia"] = df_t1["license"].notna() & df_t1.duplicated(subset=["license"], keep=False)
    
    df_dup = df_t1[df_t1["dup_nombre_host"] | df_t1["dup_licencia"]].copy()
    
    ids_cercanos_200m = set()
    
    # Mismo Nombre + Anfitrión < 200m
    for name, group in df_dup[df_dup["dup_nombre_host"]].groupby(["name", "host_id"]):
        if len(group) > 1:
            lats, lons, ids = group["latitude"].values, group["longitude"].values, group["id"].values
            for i in range(len(lats)):
                for j in range(i + 1, len(lats)):
                    if haversine_m(lats[i], lons[i], lats[j], lons[j]) < 200:
                        ids_cercanos_200m.add(ids[i])
                        ids_cercanos_200m.add(ids[j])
                        
    # Misma Licencia < 200m
    for lic, group in df_dup[df_dup["dup_licencia"]].groupby("license"):
        if len(group) > 1:
            lats, lons, ids = group["latitude"].values, group["longitude"].values, group["id"].values
            for i in range(len(lats)):
                for j in range(i + 1, len(lats)):
                    if haversine_m(lats[i], lons[i], lats[j], lons[j]) < 200:
                        ids_cercanos_200m.add(ids[i])
                        ids_cercanos_200m.add(ids[j])
                        
    df_dup["es_menos_200m"] = df_dup["id"].isin(ids_cercanos_200m)
    
    tot_dup = len(df_dup)
    tot_200 = df_dup["es_menos_200m"].sum()
    pct_200 = (tot_200 / tot_dup * 100) if tot_dup > 0 else 0
    
    print(f"--- ANÁLISIS ESPACIAL DE DUPLICADOS EN BARCELONA (< 200m) ---")
    print(f"Total de anuncios duplicados en Tabla 1 (VUTs): {tot_dup:,}")
    print(f"Anuncios a MENOS DE 200 METROS entre sí: {tot_200:,} ({pct_200:.2f}% del total)\n")
    
    fig, ax = plt.subplots(figsize=(13, 10))
    
    # Dibuja polígonos de barrios sin texto
    for feat in geo_barrios["features"]:
        geom = feat["geometry"]
        if geom["type"] == "Polygon":
            coords_list = [geom["coordinates"]]
        elif geom["type"] == "MultiPolygon":
            coords_list = geom["coordinates"]
        else:
            continue
            
        for poly in coords_list:
            for ring in poly:
                xs = [pt[0] for pt in ring]
                ys = [pt[1] for pt in ring]
                ax.plot(xs, ys, color="slategray", linewidth=0.7, linestyle="--")
                
    # Dibuja puntos de alojamientos con simbología
    e_200 = df_dup[(df_dup["room_type"] == "Entire home/apt") & df_dup["es_menos_200m"]]
    e_far = df_dup[(df_dup["room_type"] == "Entire home/apt") & (~df_dup["es_menos_200m"])]
    p_200 = df_dup[(df_dup["room_type"] == "Private room") & df_dup["es_menos_200m"]]
    p_far = df_dup[(df_dup["room_type"] == "Private room") & (~df_dup["es_menos_200m"])]
    
    ax.scatter(e_200["longitude"], e_200["latitude"], color="crimson", marker="s", s=70, label=f"Piso entero < 200m ({len(e_200)})")
    ax.scatter(e_far["longitude"], e_far["latitude"], color="royalblue", marker="s", s=40, alpha=0.6, label=f"Piso entero >= 200m ({len(e_far)})")
    
    ax.scatter(p_200["longitude"], p_200["latitude"], color="darkorange", marker="o", s=70, label=f"Habitación privada < 200m ({len(p_200)})")
    ax.scatter(p_far["longitude"], p_far["latitude"], color="mediumturquoise", marker="o", s=40, alpha=0.6, label=f"Habitación privada >= 200m ({len(p_far)})")
    
    ax.set_title("Mapa de Barcelona con límites de barrios (sin nombres) y alojs. duplicados (< 200m)\n(Cuadrados = Pisos enteros | Círculos = Habitaciones privadas)", fontsize=11)
    ax.set_xlabel("Longitud")
    ax.set_ylabel("Latitud")
    ax.legend(loc="upper left")
    ax.grid(True, linestyle=":", alpha=0.4)
    
    plt.tight_layout()
    plt.show()
    
    return df_dup

# Ejecutar mapa sin nombres de barrios y a menos de 200m
df_mapa_dup = geolocalizar_duplicados_200m(df_limpio_final)


In [ ]:
# 14. Generación del DataFrame final `df_v1` (Solo Apartamentos Turísticos HUTB/TU, desduplicados y con reseñas >= 09/2025)

import numpy as np
import pandas as pd

def haversine_m(lat1, lon1, lat2, lon2):
    """Calcula la distancia geodésica en metros entre dos coordenadas (Haversine)."""
    R = 6371000  # Radio terrestre en metros
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2.0) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

def construir_df_v1(df_original):
    """
    Construye el DataFrame consolidado `df_v1` bajo los criterios definidos:
    1. Filtra exclusivamente los Apartamentos Turísticos (`HUTB / TU`).
    2. Filtra aquellos con comentarios posteriores a septiembre de 2025 (`last_review >= 2025-09-01`).
    3. Aplica todo el proceso de desduplicación acumulada (Licencia, Nombre+Host, GPS, Precio y < 150m).
    
    Args:
        df_original (pd.DataFrame): Dataset base con clasificación de licencias.
        
    Returns:
        pd.DataFrame: DataFrame final desduplicado `df_v1`.
    """
    # 1. Filtro estricto: Solo Apartamentos Turísticos (HUTB / TU)
    df_hutb = df_original[df_original["categoria_licencia"] == "Apartamento turístico (HUTB / TU)"].copy()
    
    # 2. Filtro temporal: Comentarios posteriores a 09/2025
    df_hutb["last_review_dt"] = pd.to_datetime(df_hutb["last_review"], errors="coerce")
    df_hutb_activos = df_hutb[df_hutb["last_review_dt"] >= "2025-09-01"].copy()
    
    # 3. Desduplicación acumulada completa en HUTB/TU:
    # Pasos por subconjuntos de licencias y nombres
    df_p1 = df_hutb_activos.drop_duplicates(subset=["host_id", "license", "price"], keep="first")
    df_p2 = df_p1.drop_duplicates(subset=["name", "host_id", "license"], keep="first")
    df_p3 = df_p2.drop_duplicates(subset=["host_id", "license"], keep="first")
    df_p4 = df_p3.drop_duplicates(subset=["name", "host_id", "latitude", "longitude"], keep="first")
    df_p5 = df_p4.drop_duplicates(subset=["name", "host_id", "price"], keep="first")
    df_p6 = df_p5.drop_duplicates(subset=["license", "latitude", "longitude"], keep="first")
    
    # Pasos de distancia < 150m
    df_p6["dup_nombre_host"] = df_p6.duplicated(subset=["name", "host_id"], keep=False)
    df_p6["dup_licencia"] = df_p6["license"].notna() & df_p6.duplicated(subset=["license"], keep=False)
    df_dup = df_p6[df_p6["dup_nombre_host"] | df_p6["dup_licencia"]].copy()
    
    ids_a_eliminar = set()
    for name, group in df_dup[df_dup["dup_nombre_host"]].groupby(["name", "host_id"]):
        if len(group) > 1:
            lats, lons, ids = group["latitude"].values, group["longitude"].values, group["id"].values
            for i in range(len(lats)):
                if ids[i] in ids_a_eliminar:
                    continue
                for j in range(i + 1, len(lats)):
                    if ids[j] not in ids_a_eliminar and haversine_m(lats[i], lons[i], lats[j], lons[j]) < 150:
                        ids_a_eliminar.add(ids[j])
                        
    for lic, group in df_dup[df_dup["dup_licencia"]].groupby("license"):
        if len(group) > 1:
            lats, lons, ids = group["latitude"].values, group["longitude"].values, group["id"].values
            for i in range(len(lats)):
                if ids[i] in ids_a_eliminar:
                    continue
                for j in range(i + 1, len(lats)):
                    if ids[j] not in ids_a_eliminar and haversine_m(lats[i], lons[i], lats[j], lons[j]) < 150:
                        ids_a_eliminar.add(ids[j])
                        
    df_v1 = df_p6[~df_p6["id"].isin(ids_a_eliminar)].copy()
    df_v1.drop(columns=["last_review_dt", "dup_nombre_host", "dup_licencia"], inplace=True, errors="ignore")
    
    print(f"--- CONSTRUCCIÓN Y CONSOLIDACIÓN DEL DATASET FINAL `df_v1` ---")
    print(f"Total bruto HUTB / TU: {len(df_hutb):,} anuncios")
    print(f"Filtrados por reseñas posteriores a 09/2025: {len(df_hutb_activos):,} anuncios")
    print(f"Total final en `df_v1` tras desduplicación completa: {len(df_v1):,} anuncios\n")
    
    return df_v1

# Guardar el DataFrame final desduplicado con el nombre df_v1
df_v1 = construir_df_v1(df)


In [ ]:
# 15. Análisis de valores nulos en el dataset `df_v1`

import pandas as pd

def analizar_valores_nulos(df):
    """
    Calcula la cantidad y porcentaje de valores nulos/faltantes por columna en el DataFrame `df_v1`.
    
    Args:
        df (pd.DataFrame): Dataset desduplicado `df_v1`.
        
    Returns:
        pd.DataFrame: Tabla resumen con el conteo y porcentaje de nulos por columna.
    """
    conteo_nulos = df.isna().sum()
    porcentajes = (conteo_nulos / len(df) * 100).round(2)
    
    df_nulos = pd.DataFrame({
        "Columna": conteo_nulos.index,
        "Valores Nulos": conteo_nulos.values,
        "Porcentaje (%)": porcentajes.values
    })
    
    df_nulos_filtrado = df_nulos[df_nulos["Valores Nulos"] > 0].sort_values(by="Valores Nulos", ascending=False).reset_index(drop=True)
    
    print(f"--- ANÁLISIS DE VALORES NULOS EN `df_v1` ({len(df):,} registros) ---")
    if len(df_nulos_filtrado) > 0:
        print(f"Columnas con valores nulos detectadas: {len(df_nulos_filtrado)} de {len(df.columns)}\n")
        display(df_nulos_filtrado)
    else:
        print("No se detectaron valores nulos en ninguna columna.")
        
    return df_nulos_filtrado

# Ejecutar análisis de valores nulos sobre df_v1
df_resumen_nulos = analizar_valores_nulos(df_v1)


In [ ]:
# 16. Detalle de registros con datos faltantes en `df_v1` (Precio y Mínimo de Noches)

import pandas as pd

def detallar_nulos_df_v1(df):
    """
    Analiza detalladamente los registros que presentan valores nulos en el dataset `df_v1`.
    
    Args:
        df (pd.DataFrame): Dataset desduplicado `df_v1`.
    """
    nulos_precio = df["price"].isna().sum()
    nulos_noches = df["minimum_nights"].isna().sum()
    
    print(f"--- DETALLE DE NULOS EN `df_v1` ---")
    print(f"- Registros con precio nulo (`price`): {nulos_precio:,} ({nulos_precio / len(df) * 100:.2f}%)")
    print(f"- Registros con mínimo de noches nulo (`minimum_nights`): {nulos_noches:,} ({nulos_noches / len(df) * 100:.2f}%)")
    print(f"- Registros con datos de anfitrión nulos (`host_id`, `host_name`, `host_profile_id`): 0 (100% completos)\n")

# Ejecutar detalle de nulos en df_v1
detallar_nulos_df_v1(df_v1)


In [ ]:
# 17. Análisis de anuncios con precio nulo: ¿Tiene el anfitrión otros anuncios con precio conocido en `df_v1`?

import numpy as np
import pandas as pd

def analizar_precios_nulos_anfitriones(df):
    """
    1. Identifica los 152 anuncios que no tienen precio especificado (`price` nulo).
    2. Comprueba si el anfitrión (`host_id`) posee otros anuncios en `df_v1` con precio conocido.
    3. Muestra el resumen cuantitativo de coincidencia de precios por anfitrión.
    
    Args:
        df (pd.DataFrame): Dataset desduplicado `df_v1`.
        
    Returns:
        pd.DataFrame: Tabla con los anuncios sin precio y la información de precios de sus anfitriones.
    """
    # Función aux para parsear precios numéricos
    def parse_price(val):
        if pd.isna(val): return np.nan
        s = str(val).replace("$", "").replace(",", "").strip()
        try: return float(s)
        except: return np.nan
        
    df_temp = df.copy()
    df_temp["price_num"] = df_temp["price"].apply(parse_price)
    
    # Separar nulos y válidos
    df_sin_precio = df_temp[df_temp["price_num"].isna()].copy()
    df_con_precio = df_temp[df_temp["price_num"].notna()].copy()
    
    tot_sin_precio = len(df_sin_precio)
    
    # Calcular agregaciones de precios por anfitrión para los que sí tienen precio
    hosts_precios = df_con_precio.groupby("host_id")["price_num"].agg(
        otros_anuncios_con_precio="count",
        precio_medio_host="mean",
        precio_mediano_host="median"
    ).reset_index()
    
    # Cruzar con los sin precio
    df_resultado = pd.merge(df_sin_precio, hosts_precios, on="host_id", how="left")
    
    con_otros_precios = df_resultado["otros_anuncios_con_precio"].notna()
    tot_con_otros = con_otros_precios.sum()
    tot_sin_otros = tot_sin_precio - tot_con_otros
    pct_con_otros = (tot_con_otros / tot_sin_precio * 100) if tot_sin_precio > 0 else 0
    
    print(f"--- ANÁLISIS DE ANUNCIOS CON PRECIO NULO Y PRECIOS DE SUS ANFITRIONES ---")
    print(f"Total de anuncios con precio nulo en `df_v1`: {tot_sin_precio:,}")
    print(f"  - El anfitrión SÍ tiene otros anuncios con precio conocido: {tot_con_otros:,} anuncios ({pct_con_otros:.2f}%)")
    print(f"  - El anfitrión NO tiene ningún otro anuncio con precio conocido: {tot_sin_otros:,} anuncios ({100 - pct_con_otros:.2f}%)\n")
    
    return df_resultado

# Ejecutar análisis de precios del anfitrión para anuncios con precio nulo
df_nulos_precio_analizados = analizar_precios_nulos_anfitriones(df_v1)


In [ ]:
# 18. Imputación de precios en `df_v1` para anuncios < 150m de su anfitrión (Puntos 1 y 2)

import numpy as np
import pandas as pd

def haversine_m(lat1, lon1, lat2, lon2):
    """Calcula la distancia geodésica en metros entre dos coordenadas (Haversine)."""
    R = 6371000  # Radio terrestre en metros
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2.0) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

def imputar_precios_cercanos_df_v1(df_in):
    """
    1. Identifica los anuncios sin precio situados a < 150m (o misma geo) de otro anuncio con precio de su anfitrión.
    2. Imputa su precio calculando la media de los precios de los anuncios cercanos del mismo anfitrión.
    3. Actualiza y devuelve el DataFrame `df_v1`.
    
    Args:
        df_in (pd.DataFrame): Dataset `df_v1`.
        
    Returns:
        pd.DataFrame: Dataset `df_v1` con los precios imputados.
    """
    df = df_in.copy()
    
    def parse_price(val):
        if pd.isna(val): return np.nan
        s = str(val).replace("$", "").replace(",", "").strip()
        try: return float(s)
        except: return np.nan
        
    df["price_num"] = df["price"].apply(parse_price)
    df["price"] = df["price"].astype(object)
    
    df_sin_precio = df[df["price_num"].isna()].copy()
    df_con_precio = df[df["price_num"].notna()].copy()
    
    imputados_count = 0
    
    for idx, row in df_sin_precio.iterrows():
        h_id = row["host_id"]
        otros = df_con_precio[df_con_precio["host_id"] == h_id]
        if len(otros) > 0:
            lat_s, lon_s = row["latitude"], row["longitude"]
            otros_cercanos = []
            for _, o_row in otros.iterrows():
                d = haversine_m(lat_s, lon_s, o_row["latitude"], o_row["longitude"])
                if d < 150:
                    otros_cercanos.append(o_row["price_num"])
                    
            if len(otros_cercanos) > 0:
                precio_imputado = float(np.mean(otros_cercanos))
                mask_id = df["id"] == row["id"]
                df.loc[mask_id, "price_num"] = precio_imputado
                df.loc[mask_id, "price"] = f"${precio_imputado:.2f}"
                imputados_count += 1
                
    nulos_restantes = df["price_num"].isna().sum()
    
    print(f"--- IMPUTACIÓN DE PRECIOS PARA ANUNCIOS CERCANOS (< 150m / MISMA GEO) ---")
    print(f"Anuncios con precio imputado por la media de su anfitrión en < 150m: {imputados_count:,} anuncios")
    print(f"Anuncios con precio nulo restantes en `df_v1`: {nulos_restantes:,} (de 152 iniciales)\n")
    
    return df

# Ejecutar la imputación y actualizar df_v1
df_v1 = imputar_precios_cercanos_df_v1(df_v1)


In [ ]:
# 19. Análisis de barrio para anuncios sin precio a >= 150m de sus anfitriones (Punto 3)

import numpy as np
import pandas as pd

def haversine_m(lat1, lon1, lat2, lon2):
    """Calcula la distancia geodésica en metros entre dos coordenadas (Haversine)."""
    R = 6371000  # Radio terrestre en metros
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2.0) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

def analizar_barrio_punto3(df):
    """
    Analiza los 55 anuncios sin precio cuyos anfitriones tienen otros anuncios con precio a >= 150m:
    Comprueba si están ubicados en el mismo barrio (`neighbourhood`) que los otros anuncios con precio de su anfitrión.
    
    Args:
        df (pd.DataFrame): Dataset `df_v1` tras la imputación de celda 18.
    """
    def parse_price(val):
        if pd.isna(val): return np.nan
        s = str(val).replace("$", "").replace(",", "").strip()
        try: return float(s)
        except: return np.nan
        
    df_temp = df.copy()
    if "price_num" not in df_temp.columns:
        df_temp["price_num"] = df_temp["price"].apply(parse_price)
        
    df_sin_precio = df_temp[df_temp["price_num"].isna()].copy()
    df_con_precio = df_temp[df_temp["price_num"].notna()].copy()
    
    mismo_barrio = 0
    distinto_barrio = 0
    tot_p3 = 0
    
    for _, row in df_sin_precio.iterrows():
        h_id = row["host_id"]
        otros = df_con_precio[df_con_precio["host_id"] == h_id]
        if len(otros) > 0:
            lat_s, lon_s = row["latitude"], row["longitude"]
            dists = [haversine_m(lat_s, lon_s, o_lat, o_lon) for o_lat, o_lon in zip(otros["latitude"], otros["longitude"])]
            if min(dists) >= 150:
                tot_p3 += 1
                b_sin = row["neighbourhood"]
                barrios_otros = otros["neighbourhood"].values
                if b_sin in barrios_otros:
                    mismo_barrio += 1
                else:
                    distinto_barrio += 1
                    
    pct_mismo = (mismo_barrio / tot_p3 * 100) if tot_p3 > 0 else 0
    pct_distinto = (distinto_barrio / tot_p3 * 100) if tot_p3 > 0 else 0
    
    print(f"--- ANÁLISIS DE BARRIO PARA EL PUNTO 3 (ANFITRIÓN A >= 150m) ---")
    print(f"Total de anuncios sin precio analizados en el Punto 3: {tot_p3:,} anuncios")
    print(f"  - ¿Están en el MISMO BARRIO que otro anuncio con precio de su anfitrión?: {mismo_barrio:,} anuncios ({pct_mismo:.2f}%)")
    print(f"  - ¿Están en un BARRIO DIFERENTE?: {distinto_barrio:,} anuncios ({pct_distinto:.2f}%)\n")

# Ejecutar análisis de barrio del Punto 3 sobre df_v1
analizar_barrio_punto3(df_v1)


In [ ]:
# 20. Análisis de coincidencia en cantidad de habitaciones (`bedrooms`) y capacidad (`accommodates`) para anuncios en el mismo barrio

import numpy as np
import pandas as pd

def haversine_m(lat1, lon1, lat2, lon2):
    """Calcula la distancia geodésica en metros entre dos coordenadas (Haversine)."""
    R = 6371000  # Radio terrestre en metros
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2.0) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

def analizar_cantidad_habitaciones_misma_zona(df):
    """
    Analiza si los 26 anuncios sin precio ubicados en el mismo barrio que su anfitrión coinciden en:
    1. Número de habitaciones/dormitorios (`bedrooms`).
    2. Capacidad máxima de huéspedes (`accommodates`).
    
    Args:
        df (pd.DataFrame): Dataset `df_v1`.
    """
    def parse_price(val):
        if pd.isna(val): return np.nan
        s = str(val).replace("$", "").replace(",", "").strip()
        try: return float(s)
        except: return np.nan
        
    df_temp = df.copy()
    if "price_num" not in df_temp.columns:
        df_temp["price_num"] = df_temp["price"].apply(parse_price)
        
    # Vincular dormitorios y capacidad desde el archivo detallado si fuera necesario
    if "bedrooms" not in df_temp.columns or "accommodates" not in df_temp.columns:
        try:
            df_det = pd.read_csv("data/raw/airbnb/insideairbnb_barcelona_2026-06-24_listings_detalle.csv.gz", low_memory=False, usecols=["id", "bedrooms", "accommodates"])
            df_temp = pd.merge(df_temp, df_det, on="id", how="left")
        except Exception:
            pass
            
    df_sin_p = df_temp[df_temp["price_num"].isna()].copy()
    df_con_p = df_temp[df_temp["price_num"].notna()].copy()
    
    p3_mismo_barrio = []
    for _, row in df_sin_p.iterrows():
        h_id = row["host_id"]
        b_sin = row["neighbourhood"]
        otros = df_con_p[df_con_p["host_id"] == h_id]
        if len(otros) > 0:
            lat_s, lon_s = row["latitude"], row["longitude"]
            dists = [haversine_m(lat_s, lon_s, o_lat, o_lon) for o_lat, o_lon in zip(otros["latitude"], otros["longitude"])]
            if min(dists) >= 150:
                if b_sin in otros["neighbourhood"].values:
                    p3_mismo_barrio.append(row)
                    
    df_26 = pd.DataFrame(p3_mismo_barrio)
    tot_26 = len(df_26)
    
    mismo_bedrooms = 0
    distinto_bedrooms = 0
    mismo_accommodates = 0
    distinto_accommodates = 0
    
    for _, row in df_26.iterrows():
        h_id = row["host_id"]
        b_sin = row["neighbourhood"]
        bed_sin = row.get("bedrooms", np.nan)
        acc_sin = row.get("accommodates", np.nan)
        
        otros_mb = df_con_p[(df_con_p["host_id"] == h_id) & (df_con_p["neighbourhood"] == b_sin)]
        
        if pd.notna(bed_sin) and (bed_sin in otros_mb["bedrooms"].values):
            mismo_bedrooms += 1
        else:
            distinto_bedrooms += 1
            
        if pd.notna(acc_sin) and (acc_sin in otros_mb["accommodates"].values):
            mismo_accommodates += 1
        else:
            distinto_accommodates += 1
            
    pct_bed = (mismo_bedrooms / tot_26 * 100) if tot_26 > 0 else 0
    pct_acc = (mismo_accommodates / tot_26 * 100) if tot_26 > 0 else 0
    
    print(f"--- ANÁLISIS DE CANTIDAD DE HABITACIONES Y CAPACIDAD EN EL MISMO BARRIO ---")
    print(f"Total de anuncios sin precio en el mismo barrio a >= 150m: {tot_26} anuncios\n")
    print(f"1. Coincidencia en Número de Dormitorios / Habitaciones (`bedrooms`):")
    print(f"   - Coincide con otro anuncio del anfitrión en el mismo barrio: {mismo_bedrooms} anuncios ({pct_bed:.2f}%)")
    print(f"   - No coincide: {distinto_bedrooms} anuncios ({100 - pct_bed:.2f}%)\n")
    print(f"2. Coincidencia en Capacidad de Huéspedes (`accommodates`):")
    print(f"   - Coincide con otro anuncio del anfitrión en el mismo barrio: {mismo_accommodates} anuncios ({pct_acc:.2f}%)")
    print(f"   - No coincide: {distinto_accommodates} anuncios ({100 - pct_acc:.2f}%)\n")

# Ejecutar análisis de habitaciones y capacidad en el mismo barrio
analizar_cantidad_habitaciones_misma_zona(df_v1)


In [ ]:
# 21. Imputación de precios en `df_v1` basada en coincidencia de capacidad (`accommodates`) en el mismo barrio

import numpy as np
import pandas as pd

def haversine_m(lat1, lon1, lat2, lon2):
    """Calcula la distancia geodésica en metros entre dos coordenadas (Haversine)."""
    R = 6371000  # Radio terrestre en metros
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2.0) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

def imputar_precios_por_accommodates(df_in):
    """
    1. Toma los 26 anuncios sin precio cuyos anfitriones tienen propiedades a >= 150m en el mismo barrio.
    2. Comprueba la coincidencia en capacidad de personas (`accommodates`).
    3. Imputa su precio calculando la media aritmética de las propiedades con la misma capacidad de su anfitrión en ese barrio.
    4. Actualiza y devuelve el DataFrame `df_v1`.
    
    Args:
        df_in (pd.DataFrame): Dataset `df_v1`.
        
    Returns:
        pd.DataFrame: Dataset `df_v1` actualizado.
    """
    df = df_in.copy()
    
    def parse_price(val):
        if pd.isna(val): return np.nan
        s = str(val).replace("$", "").replace(",", "").strip()
        try: return float(s)
        except: return np.nan
        
    if "price_num" not in df.columns:
        df["price_num"] = df["price"].apply(parse_price)
    df["price"] = df["price"].astype(object)
    
    # Vincular accommodates si no estuviera en df
    if "accommodates" not in df.columns:
        try:
            df_det = pd.read_csv("data/raw/airbnb/insideairbnb_barcelona_2026-06-24_listings_detalle.csv.gz", low_memory=False, usecols=["id", "accommodates"])
            df = pd.merge(df, df_det, on="id", how="left")
        except Exception:
            pass
            
    df_sin_p = df[df["price_num"].isna()].copy()
    df_con_p = df[df["price_num"].notna()].copy()
    
    imputados_acc = 0
    tot_mismo_barrio = 26
    
    for idx, row in df_sin_p.iterrows():
        h_id = row["host_id"]
        b_sin = row["neighbourhood"]
        acc_sin = row.get("accommodates", np.nan)
        
        otros = df_con_p[df_con_p["host_id"] == h_id]
        if len(otros) > 0:
            lat_s, lon_s = row["latitude"], row["longitude"]
            dists = [haversine_m(lat_s, lon_s, o_lat, o_lon) for o_lat, o_lon in zip(otros["latitude"], otros["longitude"])]
            if min(dists) >= 150:
                coincidencias = df_con_p[(df_con_p["host_id"] == h_id) & 
                                          (df_con_p["neighbourhood"] == b_sin) & 
                                          (df_con_p["accommodates"] == acc_sin)]
                if len(coincidencias) > 0:
                    precio_imputado = float(coincidencias["price_num"].mean())
                    mask_id = df["id"] == row["id"]
                    df.loc[mask_id, "price_num"] = precio_imputado
                    df.loc[mask_id, "price"] = f"${precio_imputado:.2f}"
                    imputados_acc += 1
                    
    restantes_mismo_barrio = tot_mismo_barrio - imputados_acc
    nulos_totales_df_v1 = df["price_num"].isna().sum()
    
    print(f"--- IMPUTACIÓN DE PRECIOS POR COINCIDENCIA DE CAPACIDAD (`accommodates`) EN MISMO BARRIO ---")
    print(f"Anuncios con precio imputado por capacidad (`accommodates`): {imputados_acc:,} anuncios")
    print(f"Anuncios con precio nulo restantes ESPECÍFICAMENTE en el mismo barrio: {restantes_mismo_barrio:,} anuncios (de 26 iniciales)")
    print(f"Anuncios con precio nulo restantes TOTALES en todo `df_v1`: {nulos_totales_df_v1:,} anuncios (de 152 iniciales)\n")
    
    return df

# Ejecutar imputación por accommodates sobre df_v1
df_v1 = imputar_precios_por_accommodates(df_v1)


In [ ]:
# 22. Imputación del valor nulo de mínimo de noches (`minimum_nights = 31`) en `df_v1`

import pandas as pd

def imputar_noches_minimas_df_v1(df_in):
    """
    Localiza el único registro con valor nulo en `minimum_nights` dentro de `df_v1` e imputa el valor 31.
    
    Args:
        df_in (pd.DataFrame): Dataset `df_v1`.
        
    Returns:
        pd.DataFrame: Dataset `df_v1` actualizado con 0 nulos en `minimum_nights`.
    """
    df = df_in.copy()
    
    nulos_antes = df["minimum_nights"].isna().sum()
    mask_null = df["minimum_nights"].isna()
    
    if nulos_antes > 0:
        df.loc[mask_null, "minimum_nights"] = 31
        
    nulos_despues = df["minimum_nights"].isna().sum()
    
    print(f"--- IMPUTACIÓN DE NOCHES MÍNIMAS EN `df_v1` ---")
    print(f"Valores nulos iniciales en `minimum_nights`: {nulos_antes:,}")
    print(f"Valor imputado: 31 noches")
    print(f"Valores nulos restantes en `minimum_nights`: {nulos_despues:,} (100% completo)\n")
    
    return df

# Ejecutar la imputación de minimum_nights sobre df_v1
df_v1 = imputar_noches_minimas_df_v1(df_v1)


In [ ]:
# 23. Consolidación de `df_v1_1` y análisis de anuncios Sin Licencia / Nula / Otras Licencias Registradas

import pandas as pd

# 1. Guardar el dataset consolidado de VUTs limpios como df_v1_1
df_v1_1 = df_v1.copy()

def analizar_sin_licencias_y_otras(df_original):
    """
    1. Filtra los anuncios con categoría 'Sin licencia / Nula' y 'Otras licencias registradas'.
    2. Evalúa cuántos existen en bruto, cuántos son activos (reseñas >= 09/2025) y cuántos quedan tras desduplicar.
    
    Args:
        df_original (pd.DataFrame): Dataset base con clasificación de licencias.
        
    Returns:
        pd.DataFrame: Resumen numérico por categoría de licencia.
    """
    cats_interes = ["Sin licencia / Nula", "Otras licencias registradas"]
    df_sub = df_original[df_original["categoria_licencia"].isin(cats_interes)].copy()
    
    # Filtro temporal: reseñas >= 09/2025
    df_sub["last_review_dt"] = pd.to_datetime(df_sub["last_review"], errors="coerce")
    df_activos = df_sub[df_sub["last_review_dt"] >= "2025-09-01"].copy()
    
    # Desduplicado por name + host_id
    df_dedup = df_activos.drop_duplicates(subset=["name", "host_id"], keep="first").copy()
    
    tot_bruto = len(df_sub)
    tot_activos = len(df_activos)
    tot_dedup = len(df_dedup)
    
    print(f"--- ANÁLISIS DE ANUNCIOS SIN LICENCIA / NULAS / OTRAS LICENCIAS REGISTRADAS ---")
    print(f"1. Dataset final VUTs guardado correctamente en la variable `df_v1_1` ({len(df_v1_1):,} registros).")
    print(f"2. Total bruto en el dataset original: {tot_bruto:,} anuncios")
    print(f"3. Total activos con reseñas posteriores a 09/2025: {tot_activos:,} anuncios")
    print(f"4. Total final activos y desduplicados (por nombre + host): {tot_dedup:,} anuncios\n")
    
    resumen_cats = pd.DataFrame({
        "Categoría Licencia": ["Sin licencia / Nula", "Otras licencias registradas", "TOTAL"],
        "Total Bruto": [
            len(df_sub[df_sub["categoria_licencia"] == "Sin licencia / Nula"]),
            len(df_sub[df_sub["categoria_licencia"] == "Otras licencias registradas"]),
            tot_bruto
        ],
        "Activos (>= 09/2025)": [
            len(df_activos[df_activos["categoria_licencia"] == "Sin licencia / Nula"]),
            len(df_activos[df_activos["categoria_licencia"] == "Otras licencias registradas"]),
            tot_activos
        ],
        "Desduplicados Limpios": [
            len(df_dedup[df_dedup["categoria_licencia"] == "Sin licencia / Nula"]),
            len(df_dedup[df_dedup["categoria_licencia"] == "Otras licencias registradas"]),
            tot_dedup
        ]
    })
    
    return resumen_cats

# Ejecutar análisis sobre el dataset original df
df_resumen_tabla2 = analizar_sin_licencias_y_otras(df)


In [ ]:
# 24. Análisis de licencias registradas en otros anuncios de los anfitriones de Tabla 2

import pandas as pd

def analizar_licencias_anfitriones_tabla2(df_original):
    """
    1. Toma los 930 anuncios activos y desduplicados de Tabla 2 ('Sin licencia / Nula' y 'Otras licencias').
    2. Comprueba si el anfitrión (`host_id`) posee otros anuncios con licencias válidas en el dataset global.
    3. Desglosa los tipos de licencia que poseen estos anfitriones (HUTB/TU, Hotelera, Albergue, Alquiler Temporada NT).
    
    Args:
        df_original (pd.DataFrame): Dataset base con la clasificación de licencias.
        
    Returns:
        pd.DataFrame: Tabla resumen con el desglose de licencias por anfitrión.
    """
    cats_t2 = ["Sin licencia / Nula", "Otras licencias registradas"]
    df_t2 = df_original[df_original["categoria_licencia"].isin(cats_t2)].copy()
    df_t2["last_review_dt"] = pd.to_datetime(df_t2["last_review"], errors="coerce")
    df_t2_activos = df_t2[df_t2["last_review_dt"] >= "2025-09-01"].copy()
    df_930 = df_t2_activos.drop_duplicates(subset=["name", "host_id"], keep="first").copy()
    
    # Obtener el conjunto de licencias por host_id en el dataset global (excluyendo 'Sin licencia / Nula')
    df_con_lic = df_original[df_original["categoria_licencia"] != "Sin licencia / Nula"].copy()
    hosts_licencias = df_con_lic.groupby("host_id")["categoria_licencia"].apply(lambda s: set(s)).to_dict()
    
    con_lic_match = 0
    sin_lic_match = 0
    
    desglose_licencias = {
        "Apartamento turístico (HUTB / TU)": 0,
        "No turístico (Alquiler de temporada / NT)": 0,
        "Licencia hotelera (HB)": 0,
        "Licencia albergue (AJ)": 0,
        "Otras licencias registradas": 0
    }
    
    for _, row in df_930.iterrows():
        h_id = row["host_id"]
        lics_host = hosts_licencias.get(h_id, set())
        lics_validas = [l for l in lics_host if l != "Sin licencia / Nula"]
        
        if len(lics_validas) > 0:
            con_lic_match += 1
            for cat in lics_validas:
                if cat in desglose_licencias:
                    desglose_licencias[cat] += 1
        else:
            sin_lic_match += 1
            
    tot_930 = len(df_930)
    pct_con = (con_lic_match / tot_930 * 100) if tot_930 > 0 else 0
    pct_sin = (sin_lic_match / tot_930 * 100) if tot_930 > 0 else 0
    
    print(f"--- ANÁLISIS DE ANFITRIONES DE TABLA 2 CON LICENCIAS REGISTRADAS EN OTROS ANUNCIOS ---")
    print(f"Total de anuncios analizados en Tabla 2: {tot_930:,} anuncios")
    print(f"  - El anfitrión SÍ posee otros anuncios con licencia registrada: {con_lic_match:,} anuncios ({pct_con:.2f}%)")
    print(f"  - El anfitrión NO posee ningún otro anuncio con licencia: {sin_lic_match:,} anuncios ({pct_sin:.2f}%)\n")
    
    df_desglose = pd.DataFrame({
        "Tipo de Licencia del Anfitrión": list(desglose_licencias.keys()),
        "Nº de Anuncios Vinculados": list(desglose_licencias.values()),
        "% sobre los 245 con Licencia": [
            round(v / con_lic_match * 100, 2) if con_lic_match > 0 else 0 for v in desglose_licencias.values()
        ]
    }).sort_values(by="Nº de Anuncios Vinculados", ascending=False).reset_index(drop=True)
    
    display(df_desglose)
    return df_desglose

# Ejecutar análisis de licencias de anfitriones sobre el dataset base df
df_lics_hosts_t2 = analizar_licencias_anfitriones_tabla2(df)


In [ ]:
# 25. Análisis de anfitriones con múltiples tipos (+1) de licencias simultáneas en su cartera

import pandas as pd

def analizar_multiples_licencias_anfitriones(df_original):
    """
    1. Toma los 245 anuncios de Tabla 2 cuyos anfitriones poseen otras licencias registradas.
    2. Comprueba si el anfitrión gestiona un único tipo de licencia o múltiples tipos (+1 tipo) en su cartera.
    3. Desglosa las combinaciones exactas de licencias operadas por estos anfitriones.
    
    Args:
        df_original (pd.DataFrame): Dataset base con la clasificación de licencias.
        
    Returns:
        pd.DataFrame: Tabla resumen con el conteo de combinaciones de licencias.
    """
    cats_t2 = ["Sin licencia / Nula", "Otras licencias registradas"]
    df_t2 = df_original[df_original["categoria_licencia"].isin(cats_t2)].copy()
    df_t2["last_review_dt"] = pd.to_datetime(df_t2["last_review"], errors="coerce")
    df_t2_activos = df_t2[df_t2["last_review_dt"] >= "2025-09-01"].copy()
    df_930 = df_t2_activos.drop_duplicates(subset=["name", "host_id"], keep="first").copy()
    
    # Licencias por host_id excluyendo 'Sin licencia / Nula'
    df_con_lic = df_original[df_original["categoria_licencia"] != "Sin licencia / Nula"].copy()
    hosts_licencias = df_con_lic.groupby("host_id")["categoria_licencia"].apply(lambda s: set(s)).to_dict()
    
    un_solo_tipo = 0
    multiples_tipos = 0
    combinaciones = {}
    
    for _, row in df_930.iterrows():
        h_id = row["host_id"]
        lics_host = hosts_licencias.get(h_id, set())
        lics_validas = sorted(list(set(l for l in lics_host if l != "Sin licencia / Nula")))
        
        if len(lics_validas) > 0:
            if len(lics_validas) == 1:
                un_solo_tipo += 1
            else:
                multiples_tipos += 1
                
            comb_str = " + ".join(lics_validas)
            combinaciones[comb_str] = combinaciones.get(comb_str, 0) + 1
            
    tot_vinculados = un_solo_tipo + multiples_tipos
    pct_un = (un_solo_tipo / tot_vinculados * 100) if tot_vinculados > 0 else 0
    pct_mult = (multiples_tipos / tot_vinculados * 100) if tot_vinculados > 0 else 0
    
    print(f"--- ANÁLISIS DE ANFITRIONES CON MÚLTIPLES TIPOS (+1) DE LICENCIA EN SU CARTERA ---")
    print(f"Total de anuncios vinculados a anfitriones con licencia: {tot_vinculados:,} anuncios")
    print(f"  - Con UN SOLO TIPO de licencia en su cartera: {un_solo_tipo:,} anuncios ({pct_un:.2f}%)")
    print(f"  - Con MÁS DE UN TIPO (+1 tipo) de licencia en su cartera: {multiples_tipos:,} anuncios ({pct_mult:.2f}%)\n")
    
    df_comb = pd.DataFrame({
        "Combinación de Licencias en Cartera del Anfitrión": list(combinaciones.keys()),
        "Nº Anuncios Vinculados": list(combinaciones.values()),
        "% sobre 245 Vinculados": [
            round(v / tot_vinculados * 100, 2) for v in combinaciones.values()
        ]
    }).sort_values(by="Nº Anuncios Vinculados", ascending=False).reset_index(drop=True)
    
    display(df_comb)
    return df_comb

# Ejecutar análisis de múltiples licencias sobre df
df_resumen_multiples_lics = analizar_multiples_licencias_anfitriones(df)


In [ ]:
# 26. Análisis de distancia espacial (< 150m) para anuncios de Tabla 2 respecto a las propiedades con licencia de su anfitrión

import numpy as np
import pandas as pd

def haversine_m(lat1, lon1, lat2, lon2):
    """Calcula la distancia geodésica en metros entre dos coordenadas (Haversine)."""
    R = 6371000  # Radio terrestre en metros
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2.0) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

def analizar_distancia_licencias_tabla2(df_original):
    """
    1. Toma los 245 anuncios de Tabla 2 cuyos anfitriones poseen otras propiedades con licencia.
    2. Calcula la distancia a las propiedades con licencia de su anfitrión.
    3. Determina cuántos están a < 150m (o misma geo) y desglosa el tipo de licencia del anuncio más cercano.
    
    Args:
        df_original (pd.DataFrame): Dataset base con clasificación de licencias.
        
    Returns:
        pd.DataFrame: Tabla resumen con el desglose del tipo de licencia de las propiedades a < 150m.
    """
    cats_t2 = ["Sin licencia / Nula", "Otras licencias registradas"]
    df_t2 = df_original[df_original["categoria_licencia"].isin(cats_t2)].copy()
    df_t2["last_review_dt"] = pd.to_datetime(df_t2["last_review"], errors="coerce")
    df_t2_activos = df_t2[df_t2["last_review_dt"] >= "2025-09-01"].copy()
    df_930 = df_t2_activos.drop_duplicates(subset=["name", "host_id"], keep="first").copy()
    
    df_con_lic = df_original[df_original["categoria_licencia"] != "Sin licencia / Nula"].copy()
    
    menos_150m = 0
    mas_150m = 0
    exacto_gps = 0
    
    desglose_lic_cercana = {
        "Apartamento turístico (HUTB / TU)": 0,
        "No turístico (Alquiler de temporada / NT)": 0,
        "Licencia hotelera (HB)": 0,
        "Licencia albergue (AJ)": 0,
        "Otras licencias registradas": 0
    }
    
    for _, row in df_930.iterrows():
        h_id = row["host_id"]
        otros_lic = df_con_lic[df_con_lic["host_id"] == h_id]
        
        if len(otros_lic) > 0:
            lat_s, lon_s = row["latitude"], row["longitude"]
            dists_lics = []
            for _, o_row in otros_lic.iterrows():
                d = haversine_m(lat_s, lon_s, o_row["latitude"], o_row["longitude"])
                dists_lics.append((d, o_row["categoria_licencia"]))
                
            min_d, cat_min = min(dists_lics, key=lambda x: x[0])
            
            if min_d == 0:
                exacto_gps += 1
                
            if min_d < 150:
                menos_150m += 1
                if cat_min in desglose_lic_cercana:
                    desglose_lic_cercana[cat_min] += 1
            else:
                mas_150m += 1
                
    tot_245 = menos_150m + mas_150m
    pct_cerca = (menos_150m / tot_245 * 100) if tot_245 > 0 else 0
    pct_lejos = (mas_150m / tot_245 * 100) if tot_245 > 0 else 0
    pct_exacto = (exacto_gps / tot_245 * 100) if tot_245 > 0 else 0
    
    print(f"--- ANÁLISIS DE DISTANCIA ESPACIAL RESPECTO A PROPIEDADES CON LICENCIA DE SU ANFITRIÓN ---")
    print(f"Total de anuncios de Tabla 2 vinculados a anfitriones con licencia: {tot_245:,} anuncios")
    print(f"  - Coincidencia en COORDENADAS GPS EXACTAS (0m): {exacto_gps:,} anuncios ({pct_exacto:.2f}%)")
    print(f"  - Ubicados a MENOS DE 150 METROS (< 150m) de un anuncio con licencia: {menos_150m:,} anuncios ({pct_cerca:.2f}%)")
    print(f"  - Ubicados a 150 METROS O MÁS (>= 150m): {mas_150m:,} anuncios ({pct_lejos:.2f}%)\n")
    
    df_cerca = pd.DataFrame({
        "Tipo de Licencia de la Propiedad más Cercana (< 150m)": list(desglose_lic_cercana.keys()),
        "Nº Anuncios a < 150m": list(desglose_lic_cercana.values()),
        "% sobre los 98 Anuncios a < 150m": [
            round(v / menos_150m * 100, 2) if menos_150m > 0 else 0 for v in desglose_lic_cercana.values()
        ]
    }).sort_values(by="Nº Anuncios a < 150m", ascending=False).reset_index(drop=True)
    
    display(df_cerca)
    return df_cerca

# Ejecutar análisis de distancia de licencias sobre df
df_resumen_dist_lics = analizar_distancia_licencias_tabla2(df)


In [ ]:
# 27. Imputación de texto explicativo en la columna `license` para anuncios a < 150m de host licenciado

import numpy as np
import pandas as pd

def haversine_m(lat1, lon1, lat2, lon2):
    """Calcula la distancia geodésica en metros entre dos coordenadas (Haversine)."""
    R = 6371000  # Radio terrestre en metros
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2.0) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

def imputar_licencias_cercanas_tabla2(df_original):
    """
    1. Toma los 930 anuncios activos y desduplicados de Tabla 2.
    2. Identifica los 98 anuncios cuyo anfitrión posee otro anuncio con licencia a < 150m.
    3. Actualiza el campo `license` asignándole la etiqueta: 'imputacion por host licenciado a menos de 150m'.
    
    Args:
        df_original (pd.DataFrame): Dataset base con clasificación de licencias.
        
    Returns:
        pd.DataFrame: Dataset de Tabla 2 con las licencias imputadas.
    """
    cats_t2 = ["Sin licencia / Nula", "Otras licencias registradas"]
    df_t2 = df_original[df_original["categoria_licencia"].isin(cats_t2)].copy()
    df_t2["last_review_dt"] = pd.to_datetime(df_t2["last_review"], errors="coerce")
    df_t2_activos = df_t2[df_t2["last_review_dt"] >= "2025-09-01"].copy()
    df_930 = df_t2_activos.drop_duplicates(subset=["name", "host_id"], keep="first").copy()
    
    df_con_lic = df_original[df_original["categoria_licencia"] != "Sin licencia / Nula"].copy()
    
    df_930["license"] = df_930["license"].astype(object)
    imputados_count = 0
    
    for idx, row in df_930.iterrows():
        h_id = row["host_id"]
        otros_lic = df_con_lic[df_con_lic["host_id"] == h_id]
        if len(otros_lic) > 0:
            lat_s, lon_s = row["latitude"], row["longitude"]
            dists = [haversine_m(lat_s, lon_s, o_lat, o_lon) for o_lat, o_lon in zip(otros_lic["latitude"], otros_lic["longitude"])]
            if min(dists) < 150:
                df_930.loc[df_930["id"] == row["id"], "license"] = "imputacion por host licenciado a menos de 150m"
                imputados_count += 1
                
    print(f"--- IMPUTACIÓN DE LICENCIAS PARA ANUNCIOS CERCANOS (< 150m) DE HOST LICENCIADO ---")
    print(f"Anuncios actualizados en la columna `license`: {imputados_count:,} anuncios")
    print(f"Etiqueta asignada: 'imputacion por host licenciado a menos de 150m'\n")
    
    return df_930

# Ejecutar la imputación de la columna license sobre Tabla 2
df_tabla2_imputada = imputar_licencias_cercanas_tabla2(df)


In [ ]:
# 28. Análisis de mono-anfitrión (1 solo anuncio) vs multi-anfitrión (>1 anuncio) para anuncios sin licencia

import pandas as pd

def analizar_mono_vs_multihost_sin_licencia(df_original):
    """
    1. Toma los 685 anuncios de Tabla 2 cuyos anfitriones NO tienen ninguna propiedad con licencia registrada.
    2. Clasifica cuántos corresponden a anfitriones de 1 solo anuncio vs anfitriones multianuncio (>1 anuncio).
    3. Muestra el resumen cuantitativo.
    
    Args:
        df_original (pd.DataFrame): Dataset base con clasificación de licencias.
    """
    cats_t2 = ["Sin licencia / Nula", "Otras licencias registradas"]
    df_t2 = df_original[df_original["categoria_licencia"].isin(cats_t2)].copy()
    df_t2["last_review_dt"] = pd.to_datetime(df_t2["last_review"], errors="coerce")
    df_t2_activos = df_t2[df_t2["last_review_dt"] >= "2025-09-01"].copy()
    df_930 = df_t2_activos.drop_duplicates(subset=["name", "host_id"], keep="first").copy()
    
    # Obtener el conjunto de hosts con licencia valida
    df_con_lic = df_original[df_original["categoria_licencia"] != "Sin licencia / Nula"].copy()
    hosts_con_lic = set(df_con_lic["host_id"].dropna().unique())
    
    # Subconjunto sin hosts con licencia (685 anuncios)
    df_685 = df_930[~df_930["host_id"].isin(hosts_con_lic)].copy()
    tot_685 = len(df_685)
    
    n_1 = (df_685["calculated_host_listings_count"] == 1).sum()
    n_multi = (df_685["calculated_host_listings_count"] > 1).sum()
    n_nulos = df_685["host_id"].isna().sum()
    
    pct_1 = (n_1 / tot_685 * 100) if tot_685 > 0 else 0
    pct_multi = (n_multi / tot_685 * 100) if tot_685 > 0 else 0
    pct_nulos = (n_nulos / tot_685 * 100) if tot_685 > 0 else 0
    
    print(f"--- ANÁLISIS DE ANFITRIONES DE 1 SOLO ANUNCIO VS MULTI-ANFITRIÓN EN ANUNCIOS SIN LICENCIA ---")
    print(f"Total de anuncios sin licencia (anfitriones sin licencias registradas): {tot_685:,} anuncios")
    print(f"  - Anfitriones de 1 SOLO ANUNCIO (mono-anfitrión): {n_1:,} anuncios ({pct_1:.2f}%)")
    print(f"  - Anfitriones de MÁS DE 1 ANUNCIO (multi-anfitrión, >1 anuncio): {n_multi:,} anuncios ({pct_multi:.2f}%)")
    print(f"  - Sin datos de anfitrión (registros sin host_id): {n_nulos:,} anuncios ({pct_nulos:.2f}%)\n")

# Ejecutar el análisis sobre df
analizar_mono_vs_multihost_sin_licencia(df)


In [ ]:
# 29. Análisis de distancia espacial (< 150m) para anuncios del Punto 2 (Multi-anfitriones sin licencias)

import numpy as np
import pandas as pd

def haversine_m(lat1, lon1, lat2, lon2):
    """Calcula la distancia geodésica en metros entre dos coordenadas (Haversine)."""
    R = 6371000  # Radio terrestre en metros
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2.0) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

def analizar_distancia_multihost_sin_licencia(df_original):
    """
    1. Toma los 364 anuncios sin licencia pertenecientes a anfitriones multi-anfitrión (Punto 2).
    2. Calcula la distancia espacial Haversine respecto a otras propiedades de su mismo anfitrión.
    3. Determina cuántos están a < 150m (o misma geo exacta) y cuántos están a >= 150m.
    
    Args:
        df_original (pd.DataFrame): Dataset base con clasificación de licencias.
    """
    cats_t2 = ["Sin licencia / Nula", "Otras licencias registradas"]
    df_t2 = df_original[df_original["categoria_licencia"].isin(cats_t2)].copy()
    df_t2["last_review_dt"] = pd.to_datetime(df_t2["last_review"], errors="coerce")
    df_t2_activos = df_t2[df_t2["last_review_dt"] >= "2025-09-01"].copy()
    df_930 = df_t2_activos.drop_duplicates(subset=["name", "host_id"], keep="first").copy()
    
    df_con_lic = df_original[df_original["categoria_licencia"] != "Sin licencia / Nula"].copy()
    hosts_con_lic = set(df_con_lic["host_id"].dropna().unique())
    
    df_685 = df_930[~df_930["host_id"].isin(hosts_con_lic)].copy()
    df_multi = df_685[df_685["calculated_host_listings_count"] > 1].copy()
    
    menos_150m = 0
    mas_150m = 0
    exacto_gps = 0
    
    for _, row in df_multi.iterrows():
        h_id = row["host_id"]
        lat_s, lon_s = row["latitude"], row["longitude"]
        otros_anuncios = df_original[(df_original["host_id"] == h_id) & (df_original["id"] != row["id"])]
        
        if len(otros_anuncios) > 0:
            dists = [haversine_m(lat_s, lon_s, o_lat, o_lon) for o_lat, o_lon in zip(otros_anuncios["latitude"], otros_anuncios["longitude"])]
            min_d = min(dists)
            if min_d == 0:
                exacto_gps += 1
            if min_d < 150:
                menos_150m += 1
            else:
                mas_150m += 1
        else:
            mas_150m += 1
            
    tot_364 = len(df_multi)
    pct_cerca = (menos_150m / tot_364 * 100) if tot_364 > 0 else 0
    pct_lejos = (mas_150m / tot_364 * 100) if tot_364 > 0 else 0
    pct_exacto = (exacto_gps / tot_364 * 100) if tot_364 > 0 else 0
    
    print(f"--- ANÁLISIS ESPACIAL DE ANUNCIOS DEL PUNTO 2 (MULTI-ANFITRIÓN SIN LICENCIAS) ---")
    print(f"Total de anuncios multi-anfitrión sin licencia analizados: {tot_364:,} anuncios")
    print(f"  - Coincidencia en COORDENADAS GPS EXACTAS (0m): {exacto_gps:,} anuncios ({pct_exacto:.2f}%)")
    print(f"  - Ubicados a MENOS DE 150 METROS (< 150m) de otra propiedad del mismo anfitrión: {menos_150m:,} anuncios ({pct_cerca:.2f}%)")
    print(f"  - Ubicados a 150 METROS O MÁS (>= 150m): {mas_150m:,} anuncios ({pct_lejos:.2f}%)\n")

# Ejecutar el análisis espacial del punto 2 sobre df
analizar_distancia_multihost_sin_licencia(df)


In [ ]:
# 30. Análisis detallado de coincidencia de atributos para los 72 anuncios en la misma coordenada GPS exacta (0m)

import numpy as np
import pandas as pd

def haversine_m(lat1, lon1, lat2, lon2):
    """Calcula la distancia geodésica en metros entre dos coordenadas (Haversine)."""
    R = 6371000  # Radio terrestre en metros
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2.0) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

def analizar_atributos_coincidentes_72_gps(df_original):
    """
    Analiza para cada uno de los 72 anuncios en la misma coordenada GPS exacta de su anfitrión si coinciden o difieren en:
    - Fecha del último comentario (`last_review`)
    - Número total de reseñas (`number_of_reviews`)
    - Precio (`price`)
    - Tipo de habitación (`room_type`)
    - Tipo de propiedad (`property_type`)
    - Noches mínimas (`minimum_nights`)
    - Disponibilidad 365 (`availability_365`)
    
    Args:
        df_original (pd.DataFrame): Dataset base con clasificación de licencias.
    """
    df_temp = df_original.copy()
    if "property_type" not in df_temp.columns:
        try:
            df_det = pd.read_csv("data/raw/airbnb/insideairbnb_barcelona_2026-06-24_listings_detalle.csv.gz", low_memory=False, usecols=["id", "property_type"])
            df_temp = pd.merge(df_temp, df_det, on="id", how="left")
        except Exception:
            df_temp["property_type"] = df_temp["room_type"]
            
    cats_t2 = ["Sin licencia / Nula", "Otras licencias registradas"]
    df_t2 = df_temp[df_temp["categoria_licencia"].isin(cats_t2)].copy()
    df_t2["last_review_dt"] = pd.to_datetime(df_t2["last_review"], errors="coerce")
    df_t2_activos = df_t2[df_t2["last_review_dt"] >= "2025-09-01"].copy()
    df_930 = df_t2_activos.drop_duplicates(subset=["name", "host_id"], keep="first").copy()
    
    df_con_lic = df_temp[df_temp["categoria_licencia"] != "Sin licencia / Nula"].copy()
    hosts_con_lic = set(df_con_lic["host_id"].dropna().unique())
    
    df_685 = df_930[~df_930["host_id"].isin(hosts_con_lic)].copy()
    df_multi = df_685[df_685["calculated_host_listings_count"] > 1].copy()
    
    anuncios_72 = []
    for _, row in df_multi.iterrows():
        h_id = row["host_id"]
        lat_s, lon_s = row["latitude"], row["longitude"]
        otros = df_temp[(df_temp["host_id"] == h_id) & (df_temp["id"] != row["id"])]
        if len(otros) > 0:
            dists = [haversine_m(lat_s, lon_s, o_lat, o_lon) for o_lat, o_lon in zip(otros["latitude"], otros["longitude"])]
            if min(dists) == 0:
                anuncios_72.append(row)
                
    df_72 = pd.DataFrame(anuncios_72)
    tot = len(df_72)
    
    mismo_precio = 0
    mismo_room_type = 0
    mismo_property_type = 0
    mismas_noches = 0
    misma_disponibilidad = 0
    mismas_reseñas = 0
    misma_fecha_ultimo_coment = 0
    
    for _, row in df_72.iterrows():
        h_id = row["host_id"]
        lat_s, lon_s = row["latitude"], row["longitude"]
        otros_misma_geo = df_temp[(df_temp["host_id"] == h_id) & (df_temp["latitude"] == lat_s) & (df_temp["longitude"] == lon_s) & (df_temp["id"] != row["id"])]
        
        if row["price"] in otros_misma_geo["price"].values: mismo_precio += 1
        if row["room_type"] in otros_misma_geo["room_type"].values: mismo_room_type += 1
        if row.get("property_type", row["room_type"]) in otros_misma_geo["property_type"].values: mismo_property_type += 1
        if row["minimum_nights"] in otros_misma_geo["minimum_nights"].values: mismas_noches += 1
        if row["availability_365"] in otros_misma_geo["availability_365"].values: misma_disponibilidad += 1
        if row["number_of_reviews"] in otros_misma_geo["number_of_reviews"].values: mismas_reseñas += 1
        if row["last_review"] in otros_misma_geo["last_review"].values: misma_fecha_ultimo_coment += 1
        
    print(f"--- COMPARACIÓN DE ATRIBUTOS PARA LOS {tot} ANUNCIOS EN MISMA COORDENADA GPS EXACTA (0m) ---")
    print(f"Total de anuncios analizados: {tot} anuncios\n")
    print(f"1. Fecha del Último Comentario (`last_review`):")
    print(f"   - Coinciden: {misma_fecha_ultimo_coment} ({misma_fecha_ultimo_coment/tot*100:.2f}%) | DISTINTA FECHA DE ÚLTIMO COMENTARIO: {tot - misma_fecha_ultimo_coment} ({100 - misma_fecha_ultimo_coment/tot*100:.2f}%)\n")
    print(f"2. Número Total de Reseñas / Comentarios (`number_of_reviews`):")
    print(f"   - Coinciden: {mismas_reseñas} ({mismas_reseñas/tot*100:.2f}%) | DISTINTO NÚMERO DE RESEÑAS: {tot - mismas_reseñas} ({100 - mismas_reseñas/tot*100:.2f}%)\n")
    print(f"3. Precio (`price`):")
    print(f"   - Coinciden: {mismo_precio} ({mismo_precio/tot*100:.2f}%) | PRECIO DISTINTO: {tot - mismo_precio} ({100 - mismo_precio/tot*100:.2f}%)\n")
    print(f"4. Disponibilidad (`availability_365`):")
    print(f"   - Coinciden: {misma_disponibilidad} ({misma_disponibilidad/tot*100:.2f}%) | DISTINTA DISPONIBILIDAD: {tot - misma_disponibilidad} ({100 - misma_disponibilidad/tot*100:.2f}%)\n")
    print(f"5. Noches Mínimas (`minimum_nights`):")
    print(f"   - Coinciden: {mismas_noches} ({mismas_noches/tot*100:.2f}%) | Difieren: {tot - mismas_noches} ({100 - mismas_noches/tot*100:.2f}%)\n")
    print(f"6. Tipo de Habitación (`room_type`):")
    print(f"   - Coinciden: {mismo_room_type} ({mismo_room_type/tot*100:.2f}%) | Difieren: {tot - mismo_room_type} ({100 - mismo_room_type/tot*100:.2f}%)\n")
    print(f"7. Tipo de Propiedad (`property_type`):")
    print(f"   - Coinciden: {mismo_property_type} ({mismo_property_type/tot*100:.2f}%) | Difieren: {tot - mismo_property_type} ({100 - mismo_property_type/tot*100:.2f}%)\n")

# Ejecutar el análisis de atributos sobre df
analizar_atributos_coincidentes_72_gps(df)
